# Reproducing Grosmark & Buzsáki (2016) on DANDI:000044 and CRCNS hc-11

Paper: Grosmark, A. D. & Buzsáki, G. (2016). Diversity in neural firing dynamics supports both rigid and learned hippocampal sequences. *Science* 351, 1440–1443, doi:10.1126/science.aad1935.

Datasets, both used:

- DANDI:000044, version 0.261106.0236, the version this attempt was asked to use (it is the version named in the prompt for this run; the dandiset's NWB files are streamed with remfile and h5py). It is used for the LFP (sharp wave-ripple detection), the lab's lap directions (trials), the automatic buzcode state scoring, and as a check on the units, spike times and state epochs.
- CRCNS hc-11 (doi:10.6080/K0862DC5), unversioned. The sessInfo files in `NoveltySessInfoMatFiles.tar.gz` (MD5 `2666ee13c8c1e3609c08c6f4ba444d11` in the data set's `checksums.md5`) are streamed from the public bucket and give the spike times, cell types, position and the lab's state epochs. The per-session tarballs `<session>.tar.gz` (0.8 to 2.3 GB each) are streamed for the PCA features and cluster ids of Figure S11; the LFP and waveform tarballs of hc-11 are not read.

The claims file `claims/grosmark-2016.yaml` was written from the NIH author manuscript (PMC4919122) with its supplement. The claims are tested on the eight sessions; both datasets hold the same recordings (checked below), which are counted once.

Shuffle counts. The paper uses 1,000 null templates per event and 1,000 cell-specific shuffles for the PCC. A full run with 1,000 of each took over an hour for five of the eight sessions while preparing, so this notebook uses 200 null templates per event (`N_NULL`) and 100 cell-specific shuffles (`N_PCC`); one session is rescored with 1,000 of each to show the effect. The reruns of Figures 2D, 2E, 4D, 4E and S12 are batched and use 200, 100 or 50 null templates (below). This is recorded as a deviation.

| Claim | Tested here |
|---|---|
| sessions-and-tracks, place-cell-count | yes |
| fig1c-above-chance, fig1c-post-gt-pre, fig1c-maze-gt-pre, figs5b-abs-r-above-null, figs5e-duration-unchanged, figs5f-more-participation-post | yes |
| figs9b-pcc-post-higher, fig2b-strong-contributors, fig2-pre-post-overlap, rigid-plastic-counts, figs19c-pcc-change-per-cell | yes |
| fig2d-shuffle-high-dpcc, figs12-high-dpcc-exclusion, fig2e-remove-plastic, fig4d-slow-sequences, fig4e-gain-sequences | yes, under all six readings of Y and D, with reduced shuffles |
| figs2-track-distribution, fig3a-plastic-specificity, fig3b-plastic-improve, figs13-outfield-rate-falls | yes |
| fig4a-rigid-faster, figs14-plastic-ripple-gain, figs14-plastic-ripple-bursting, figs16ab-plastic-synchrony-increase | yes |
| rate-split-medians, figs16c-low-rate-coding, fig4c-pre-fast-coactivation, fig4c-slow-coactivation-increase, fig4c-slow-replay | yes |
| figs16de-more-ripples-post, figs17-coactivity-preservation, figs17d-rigid-plastic-coactivity, figs18-temporal-bias | yes |
| figs11-cluster-quality-similar | yes (cluster quality and stability; the waveforms are not tested) |

In [ ]:
import os, re, time, zlib, resource
os.environ.setdefault("NUMBA_NUM_THREADS", "4")
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import grosmark as g

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
KEY = "grosmark-2016"
def seed(claim, k=0):
    return zlib.crc32(f"{KEY}/{claim}".encode()) + k
T0 = time.time()

## Loading

The CRCNS sessInfo files are streamed and parsed; each session's NWB file is opened on DANDI. `process_session` (helper) computes, per session: speed and direction, place fields per running direction, the LFP ripples, the population synchrony events, and the Bayesian replay scores with the PCC of every participating cell.

In [ ]:
S = g.load_crcns_sessions()
assets = {a["path"].split("ses-")[1].split("_")[0].replace("-", "_"): a for a in g.dandi_assets()}
print("CRCNS sessions:", list(S))
print("DANDI assets:", {k: (v["path"], v["size"]) for k, v in assets.items()})
print(f"{time.time() - T0:.0f} s")

In [ ]:
NWB, R = {}, {}
rng_pf = np.random.default_rng(seed("place-cell-count"))
rng_shuf = np.random.default_rng(seed("fig1c-above-chance"))
for name in S:
    f = g.open_nwb(assets[name]["asset_id"])
    NWB[name] = g.read_nwb_meta(f)
    R[name] = g.process_session(name, S[name], f, NWB[name], rng_pf, rng_shuf)
    f.close()
print(f"{time.time() - T0:.0f} s, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

## Data check

In [ ]:
rows = []
for name, s in S.items():
    n = NWB[name]
    E = s["epochs"]
    rows.append({"session": name, "subject": n["subject"], "maze": s["maze"],
                 "PRE_h": np.diff(E["PREEpoch"][0])[0] / 3600, "MAZE_min": np.diff(E["MazeEpoch"][0])[0] / 60,
                 "POST_h": np.diff(E["POSTEpoch"][0])[0] / 3600,
                 "order_ok": E["PREEpoch"][0][1] == E["MazeEpoch"][0][0] and E["MazeEpoch"][0][1] == E["POSTEpoch"][0][0],
                 "pyr": len(s["pyr_ids"]), "int": len(s["int_ids"]), "nwb_units": len(n["unit_ids"]),
                 "nwb_pyr": int(np.sum(n["cell_type"] == "pyramidal cell")),
                 "pos_rate_Hz": 1 / np.median(np.diff(s["pos_t"])), "pos1d_max_m": np.nanmax(s["pos_1d"]),
                 "lfp_rate": n["lfp_rate"], "lfp_samples": n["lfp_shape"][0],
                 "lfp_dur_s": n["lfp_shape"][0] / n["lfp_rate"], "sessDuration_s": s["duration"],
                 "last_spike_s": max(sp.max() for sp in s["spikes"].values()),
                 "nwb_trials": len(n["trials"][0]) if "trials" in n else 0})
SAN = pd.DataFrame(rows)
print(SAN.to_string())
print("subjects:", SAN.subject.nunique(), "sessions:", len(SAN), "pyramidal:", SAN.pyr.sum(), "interneurons:", SAN["int"].sum())
# units and spike times: CRCNS sessInfo against the DANDI NWB units
for name, s in S.items():
    n = NWB[name]
    same_ids = set(s["spikes"]) == set(int(u) for u in n["unit_ids"])
    same_spk = all(np.array_equal(np.sort(n["spikes"][u]), s["spikes"][u]) for u in s["spikes"])
    pyr_n = set(int(u) for u, c in zip(n["unit_ids"], n["cell_type"]) if c == "pyramidal cell")
    lab, a, b = n["sleep_states"]
    same_states = all(np.allclose(np.sort(a[lab == k]), np.sort(s["states"][k][:, 0])) for k in s["states"])
    print(f"{name}: unit ids equal {same_ids}, spike times equal {same_spk}, pyramidal sets equal {pyr_n == set(s['pyr_ids'])}, "
          f"lab states equal {same_states}, units with unsorted spikes {s['unsorted_units']}")
# the variables used
sp = np.concatenate([R[nm]["speed"][~np.isnan(R[nm]["speed"])] for nm in R])
print(f"speed (m/s): median {np.median(sp):.3f}, 99th percentile {np.percentile(sp, 99):.3f}, max {sp.max():.2f}")
for nm, r in R.items():
    off = r["off"]
    print(nm, "offline S-a (s): PRE %.0f MAZE %.0f POST %.0f;" % tuple(g.total(off["S-a"][e]) for e in ["PRE", "MAZE", "POST"]),
          "S-b NREM: PRE %.0f POST %.0f;" % (g.total(off["S-b"]["PRE"]), g.total(off["S-b"]["POST"])),
          f"yoke T {r['yoke']['T']:.0f} s; LFP channel {r['lfp']['channel']}, ripples z>=3 {len(r['lfp']['peak_t'])}, "
          f"z>=5 {int(np.sum(r['lfp']['peak_z'] >= 5))}; population events {r['pop_events_n']}, with an LFP ripple "
          f"{100 * r['pop_events_with_lfp_frac']:.1f}%")

Two checks on the clocks. The LFP (from DANDI) and the spikes (from CRCNS) are on one clock if the population synchrony events hold LFP ripple peaks far more often than when the ripple peaks are shifted in time. Position and spikes are on one clock if position decoded from the spikes during running is better than with the position shifted.

In [ ]:
rows = []
for nm, r in R.items():
    s = S[nm]
    pk = r["lfp"]["peak_t"]
    ev = r["events"]["t"]
    fr = {}
    for shift in [0.0, 0.5, 2.0, -2.0]:
        a = np.searchsorted(pk + shift, ev[:, 0]); b = np.searchsorted(pk + shift, ev[:, 1], side="right")
        fr[shift] = np.mean(b > a)
    # position decoding during running, true against shifted position
    d0 = g.directions(s)[0]
    dd = r["dirs"][d0]
    pcs = dd["place_cells"]; idx = [r["pyr"].index(u) for u in pcs]
    f_ = dd["sm"][idx] + g.EPS_RATE
    ok = g.run_mask(s, d0, r["speed"], r["v1"])
    t = s["pos_t"]
    err = {}
    tb = t[ok][::10]
    for shift in [0.0, 30.0]:
        e_ = []
        for tt in tb[:400]:
            n = np.array([np.sum((s["spikes"][u] >= tt - 0.25) & (s["spikes"][u] < tt + 0.25)) for u in pcs])
            ll = (n[:, None] * np.log(f_)).sum(0) - 0.5 * f_.sum(0)
            k = np.searchsorted(t, tt + shift)
            if k >= len(t) or np.isnan(s["pos_1d"][k]):
                continue
            e_.append(abs(np.argmax(ll) * g.BIN_CM / 100 - s["pos_1d"][k]))
        err[shift] = np.median(e_) if e_ else np.nan
    rows.append({"session": nm, "events with LFP peak, true": fr[0.0], "shift +0.5 s": fr[0.5], "shift +2 s": fr[2.0],
                 "shift -2 s": fr[-2.0], "decoding error m, true": err[0.0], "position shifted 30 s": err[30.0]})
print(pd.DataFrame(rows).round(3).to_string())

In [ ]:
# Position is tracked only within the maze epoch, so the running intervals of the coactivity, temporal-bias and
# cluster-quality analyses lie inside MAZE (verifier round 1).
for nm, s in S.items():
    t = s["pos_t"]; mz = s["epochs"]["MazeEpoch"][0]; sp = R[nm]["speed"]
    out = (t < mz[0]) | (t > mz[1])
    print(f"{nm}: position {t[0]:.1f} to {t[-1]:.1f} s, MazeEpoch {mz[0]:.1f} to {mz[1]:.1f} s; samples outside MazeEpoch {int(out.sum())}, "
          f"running >= 5 cm/s outside {int(np.sum(out & (sp >= g.RUN_SPEED)))}; 1D position {np.nanmin(s['pos_1d']):.3f} to {np.nanmax(s['pos_1d']):.3f} m")
# The LFP channel: ripple-band power of the 16 channels of its column on a NREM minute, and the ripple-triggered LFP
# (+-100 ms around the z >= 5 peaks of that minute) on the chosen channel: its dominant frequency and the negative
# sharp wave (mean over -20 to +20 ms against the mean over the window's ends).
for nm, r in R.items():
    L_ = r["lfp"]; tr = L_["trig_lfp"]; fs = L_["fs"]
    pw = L_["col_power"]; pw_ok = pw[pw >= 0]
    f_, P_ = __import__("scipy").signal.periodogram(tr - tr.mean(), fs, nfft=4096)
    band = (f_ >= 100) & (f_ <= 300)
    w = len(tr) // 2; k20 = int(0.02 * fs)
    sw = tr[w - k20:w + k20].mean() - np.r_[tr[:k20], tr[-k20:]].mean()
    print(f"{nm}: channel {L_['channel']}, ripple power {pw[L_['col_channel_index']] / np.median(pw_ok):.2f} x the column median; "
          f"triggered LFP over {L_['trig_n']} ripples: peak frequency {f_[band][np.argmax(P_[band])]:.0f} Hz, "
          f"peak-to-trough {np.ptp(tr[w - k20:w + k20]) * 1e6:.0f} uV, sharp wave {sw * 1e6:+.0f} uV")

The candidate ripple events above are those that already contain an LFP ripple, so the true-time fraction is 1 by construction; the shifted fractions show how often a ripple peak falls in an event by chance.

In [ ]:
for nm, s in S.items():
    last = max(sp.max() for sp in s["spikes"].values())
    n_after = sum(int(np.sum(sp > s["duration"])) for sp in s["spikes"].values())
    print(f"{nm}: sessDuration {s['duration']:.3f} s, last spike {last:.3f} s, spikes after sessDuration {n_after}, "
          f"POST epoch end {s['epochs']['POSTEpoch'][0][1]:.3f} (NWB {NWB[nm]['epochs']['POST'][1]:.3f}), "
          f"LFP end {NWB[nm]['lfp_shape'][0] / NWB[nm]['lfp_rate']:.3f}, last lab state end {max(v[:, 1].max() for v in s['states'].values()):.1f}")

In [ ]:
for nm, s in S.items():
    dt = np.diff(s["pos_t"])
    print(f"{nm}: position sampling {1 / np.median(dt):.3f} Hz (Methods: 39.69 Hz); 1D position defined in "
          f"{100 * np.mean(~np.isnan(s['pos_1d'])):.1f}% of samples; 1D range {np.nanmin(s['pos_1d']):.2f} to {np.nanmax(s['pos_1d']):.2f} m")
# lap directions: the NWB trials table (the sessInfo files have none)
for nm in S:
    tr = NWB[nm].get("trials")
    dirs_ = {} if tr is None else {str(k): int(v) for k, v in zip(*np.unique(tr[0], return_counts=True))}
    print(f"{nm} ({S[nm]['maze']}): NWB trials {0 if tr is None else len(tr[1])}, directions {dirs_}")


## sessions-and-tracks

> Recordings were made in four rats, in eight novel exploration sessions, each a PRE rest and sleep epoch in the home cage, a MAZE epoch on one of two linear tracks or a circular track in a novel room, and a POST rest and sleep epoch in the home cage.

In [ ]:
print(SAN[["session", "subject", "maze", "PRE_h", "MAZE_min", "POST_h", "order_ok", "pos1d_max_m"]].to_string())
print("rats:", SAN.subject.nunique(), "sessions:", len(SAN), "all PRE, MAZE, POST in order:", SAN.order_ok.all())
print(SAN.maze.value_counts().to_string())
lin = SAN.maze.str.contains("Linear").sum(); circ = SAN.maze.str.contains("Circular").sum()
print(f"linear {lin}, circular {circ}")

**Result: reproduced.** Both datasets hold 4 rats and 8 sessions, each with PRE, MAZE and POST epochs in this order. Five MAZE epochs are on linear tracks (four 1.6 m, one 2 m, as linearised) and three on the circular track (linearised 2.84 to 2.90 m).

## place-cell-count

> Of the 562 well isolated CA1 pyramidal cells recorded in the eight sessions, 491 had place fields on the novel maze, counted separately for each running direction on the linear mazes, which is 54% of pyramidal cells by running direction.

In [ ]:
rows = []
for nm, r in R.items():
    for d, dd in r["dirs"].items():
        rows.append({"session": nm, "dir": d, "pyr": len(r["pyr"]), "place_cells": len(dd["place_cells"])})
PC = pd.DataFrame(rows)
print(PC.to_string())
n_pc = PC.place_cells.sum(); n_pyr = SAN.pyr.sum(); n_pyrdir = PC.pyr.sum()
uniq = sum(len(set().union(*[set(dd["place_cells"]) for dd in r["dirs"].values()])) for r in R.values())
print(f"place cells per running direction {n_pc}; pyramidal cells {n_pyr}; pyramidal cells x directions {n_pyrdir}")
print(f"reading (a) {100 * n_pc / n_pyrdir:.1f}%; reading (b) {100 * n_pc / n_pyr:.1f}%; distinct cells with a field in some direction {uniq}")
for nm, r in R.items():
    print(nm, {d: len(dd["place_cells"]) for d, dd in r["dirs"].items()})
# Variants (verifier round 1): the 5 cm kernel read as its full width at half maximum (SD 5 / 2.355 cm), and the
# 1 Hz in-field peak rule dropped; same place-field test otherwise.
rng_v = np.random.default_rng(seed("place-cell-count", 1))
SD_W = 5 / 2.3548
W_PC = {}   # place cells under the width reading, used again for fig2b-strong-contributors
rows = []
for nm, r in R.items():
    for d, dd in r["dirs"].items():
        occ, counts = dd["occ"], dd["counts"]
        W_PC[(nm, d)] = [u for i, u in enumerate(r["pyr"]) if g.place_field_test(occ, counts[i], rng_v, sd_cm=SD_W)[0]]
        n_w = len(W_PC[(nm, d)])
        n_p = sum(g.place_field_test(occ, counts[i], rng_v, min_peak=0.0)[0] for i in range(len(counts)))
        rows.append({"session": nm, "dir": d, "pyr": len(counts), "SD_5cm": len(dd["place_cells"]), "FWHM_5cm": n_w, "no_1Hz_rule": n_p})
PCV = pd.DataFrame(rows)
print(PCV.to_string())
tot = PCV[["pyr", "SD_5cm", "FWHM_5cm", "no_1Hz_rule"]].sum()
for c in ["SD_5cm", "FWHM_5cm", "no_1Hz_rule"]:
    print(f"{c}: {tot[c]} place cells per running direction (442 to 540 within tolerance), {100 * tot[c] / tot.pyr:.1f}% of {tot.pyr} "
          f"pyramidal cells x directions (48.6 to 59.4% within tolerance)")

**Result: not reproduced, resting on the claims file's reading of the 5 cm kernel as its SD.** The datasets hold 562 pyramidal cells, as the paper states, but the place-field method finds 573 place cells per running direction against 491 (tolerance 442 to 540), 63.0% of pyramidal cells by running direction (reading a; 54% expected within 5.4) and 102.0% of the 562 (reading b). With the kernel read as its full width at half maximum the counts are 507 and 55.7%, both within tolerance; without the 1 Hz peak rule they are 708 and 77.8%. The count also rests on the running direction from the sign of the 1D velocity and on the lab's 1D position, defined only on the track outside the reward areas (it runs from 0 to 1.6 or 2.0 m on the linear tracks and to 2.84 to 2.90 m on the circular one). Carried into the classification (cell `kernel-width-classification`), the width reading also brings the counts of fig2b-strong-contributors and rigid-plastic-counts within tolerance under D-a. The claims file's opening comment says the choice "should not change outcomes"; here it does, and this is asked of a reader (questions/grosmark-2016.yaml).

## Ripple events and sequence scores

All candidate ripple events of every reading are scored once; a reading is a subset. Readings: Y (yoking: Y-a last T of PRE and first T of POST; Y-b first T of each; Y-c T of offline time), L (LFP ripple threshold 3 or 5 SD), S (S-a the lab's states; S-b buzcode NREM), D (D-a both direction templates kept; D-b the direction chosen by the direction decoder of equation 6). D-a is the main reading for every claim: the Methods (Bayesian Replay Analysis) say "For linear track sessions Bayesian analysis was performed independently for the left and right direction of movement", and the Figure S9C legend assigns each event to one direction only for the forward and reverse PCC. Every claim that uses the rigid and plastic split is also shown with the groups of D-b.

In [ ]:
ET = g.event_table(R)
ST = g.score_table(R, ET)
print("candidate ripple events:", len(ET), "scored (event, direction) rows:", len(ST))
print(ST.groupby(["session", "dir", "epoch"]).size().unstack().to_string())
READ = [(Y, L, S_, D) for Y in g.READINGS_Y for L in g.READINGS_L for S_ in g.READINGS_S for D in g.READINGS_D]
def sel(df, Y="Y-a", L="L-a", S_="S-a", D="D-a"):
    return df[g.reading_mask(df, Y, L, S_, D)]
print(f"{time.time() - T0:.0f} s")

## fig1c-above-chance

> Ripple-event sequence scores (rZ) are above chance in each of PRE, MAZE and POST.

In [ ]:
rows = []
for rd in READ:
    x = sel(ST, *rd)
    row = dict(zip("YLSD", rd))
    for ep in ["PRE", "MAZE", "POST"]:
        v = x[x.epoch == ep].rz
        row[f"n_{ep}"] = len(v); row[f"mean_{ep}"] = v.mean(); row[f"p_{ep}"] = stats.wilcoxon(v).pvalue if len(v) else np.nan
    rows.append(row)
F1 = pd.DataFrame(rows)
print(F1.to_string(float_format=lambda v: f"{v:.3g}"))
allpos = ((F1[["mean_PRE", "mean_MAZE", "mean_POST"]] > 0).all(axis=1) & (F1[["p_PRE", "p_MAZE", "p_POST"]] < 0.05).all(axis=1))
print("readings with rZ above zero at p < 0.05 in all three epochs:", int(allpos.sum()), "of", len(F1))
print("largest p over readings:", F1[["p_PRE", "p_MAZE", "p_POST"]].max().to_dict())

**Result: reproduced.** Mean rZ is above zero in PRE (0.233), MAZE (0.652) and POST (0.311) under the main reading, with signed-rank p below 1e-12 in every epoch under all 24 readings.

## fig1c-post-gt-pre and fig1c-maze-gt-pre

> Sequence scores (rZ) of ripple events are higher in POST sleep than in PRE sleep (replay).
> Sequence content is higher in MAZE immobility events than in PRE sleep events.

Kruskal-Wallis, then Tukey-Kramer on mean ranks (the post hoc test MATLAB's `multcompare` applies to `kruskalwallis`); Tukey-Kramer on the values is shown as a variant.

In [ ]:
rows = []
for rd in READ:
    x = sel(ST, *rd)
    G = {ep: x[x.epoch == ep].rz.values for ep in ["PRE", "MAZE", "POST"]}
    kw, tk = g.tukey_kramer_ranks(G)
    tv = g.tukey_kramer_values(G)
    rows.append({**dict(zip("YLSD", rd)), "mean_PRE": G["PRE"].mean(), "mean_MAZE": G["MAZE"].mean(), "mean_POST": G["POST"].mean(),
                 "KW_p": kw, "POSTvPRE_p": tk[("PRE", "POST")][1], "MAZEvPRE_p": tk[("PRE", "MAZE")][1],
                 "MAZEvPOST_p": tk[("MAZE", "POST")][1], "POSTvPRE_p_values": tv[("PRE", "POST")][1],
                 "MAZEvPRE_p_values": tv[("PRE", "MAZE")][1]})
F2 = pd.DataFrame(rows)
print(F2.to_string(float_format=lambda v: f"{v:.3g}"))
post_ok = (F2.mean_POST > F2.mean_PRE) & (F2.POSTvPRE_p < 0.05)
maze_ok = (F2.mean_MAZE > F2.mean_PRE) & (F2.MAZEvPRE_p < 0.05)
print("POST > PRE (Tukey-Kramer on ranks, p < 0.05):", int(post_ok.sum()), "of", len(F2), "readings")
print("MAZE > PRE (Tukey-Kramer on ranks, p < 0.05):", int(maze_ok.sum()), "of", len(F2), "readings")
print("on values: POST > PRE", int(((F2.mean_POST > F2.mean_PRE) & (F2.POSTvPRE_p_values < 0.05)).sum()),
      "MAZE > PRE", int(((F2.mean_MAZE > F2.mean_PRE) & (F2.MAZEvPRE_p_values < 0.05)).sum()))
# per animal, main reading
x = sel(ST)
x = x.assign(animal=x.session.str.split("_").str[0])
RAT = {"Buddy": "A", "Gatsby": "B", "Achilles": "C", "Cicero": "D"}   # hc-11 data description, rat A to D
pa = x.groupby(["animal", "epoch"]).rz.agg(["mean", "count"]).unstack()
print(pa.round(3).to_string())
for a_, gx in x.groupby("animal"):
    G = {ep: gx[gx.epoch == ep].rz.values for ep in ["PRE", "MAZE", "POST"]}
    kw, tk = g.tukey_kramer_ranks(G)
    print(f"{a_} (rat {RAT[a_]}): POST-PRE mean {G['POST'].mean() - G['PRE'].mean():+.3f} (p {tk[('PRE', 'POST')][1]:.3g}); "
          f"MAZE-PRE {G['MAZE'].mean() - G['PRE'].mean():+.3f} (p {tk[('PRE', 'MAZE')][1]:.3g}); highest epoch {max(G, key=lambda k: G[k].mean())}")
# bootstrap of the POST - PRE change of |r| against the null change (Figure S5B, right)
rng = np.random.default_rng(seed("fig1c-post-gt-pre"))
pre, post = x[x.epoch == "PRE"], x[x.epoch == "POST"]
d_obs = post.absr.mean() - pre.absr.mean(); d_null = post.null_mean.mean() - pre.null_mean.mean()
ci = g.boot_diff_ci(post.absr.values - post.null_mean.values, pre.absr.values - pre.null_mean.values, rng)
print(f"POST-PRE |r| change {d_obs:.4f}, null change {d_null:.4f}; bootstrap 95% CI of (observed - null) change {ci.round(4)}")

**Result: reproduced.** Mean POST rZ (0.311) exceeds mean PRE rZ (0.233) under the main reading; Kruskal-Wallis then Tukey-Kramer gives POST above PRE at p < 0.05 under all 24 readings, on ranks and on values. By animal, POST exceeds PRE in all four, significantly in Achilles (rat C, p 8.69e-11) and Cicero (rat D, p 0.0275) and not in Buddy (rat A, p 0.259) or Gatsby (rat B, p 0.169). Figure S6 shows the increase in every animal but rat B, so rat B agrees with the paper and rat A, which Figure S6 shows with an increase, does not reach significance here. The observed POST - PRE change of |r| (0.0149) exceeds the null change (0.0046), bootstrap interval of the difference 0.0078 to 0.0127 (0.0078 to 0.0125 from another bootstrap draw in figs5b-abs-r-above-null).

**Result: reproduced.** Mean MAZE rZ (0.652) exceeds mean PRE rZ (0.233), Tukey-Kramer p < 0.05 under all 24 readings; MAZE is the highest epoch in each of the four animals, and MAZE exceeds POST under all readings.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
for ep, c in [("PRE", "tab:blue"), ("MAZE", "tab:green"), ("POST", "tab:red")]:
    v = np.sort(x[x.epoch == ep].rz.values)
    ax.plot(v, np.linspace(0, 1, len(v)), c=c, label=f"{ep} (n={len(v)})")
ax.set_xlim(-3, 8); ax.set_xlabel("rZ"); ax.set_ylabel("cumulative fraction"); ax.legend()
print("rZ range", x.rz.min().round(2), x.rz.max().round(2), "points outside the x axis:", int(((x.rz < -3) | (x.rz > 8)).sum()))
plt.show()

## figs5b-abs-r-above-null

> Absolute weighted correlations of ripple events exceed those of the shuffled null in PRE, MAZE and POST, and the POST minus PRE increase is larger in the observed than in the null data.

In [ ]:
rows = []
for rd in READ:
    x5 = sel(ST, *rd)
    rng = np.random.default_rng(seed("figs5b-abs-r-above-null"))
    row = dict(zip("YLSD", rd))
    for ep in ["PRE", "MAZE", "POST"]:
        v = x5[x5.epoch == ep]
        ns_ = np.concatenate(v.null_sample.values)
        row[f"p_{ep}"] = stats.ranksums(v.absr, ns_, alternative="greater").pvalue
        row[f"gt_{ep}"] = v.absr.mean() > ns_.mean()
        row[f"med_obs_{ep}"] = v.absr.median(); row[f"med_null_{ep}"] = np.median(ns_)
        row[f"p_evmean_{ep}"] = stats.ranksums(v.absr, v.null_mean, alternative="greater").pvalue
    pre5, post5 = x5[x5.epoch == "PRE"], x5[x5.epoch == "POST"]
    ia = rng.integers(0, len(post5), (1000, len(post5))); ib = rng.integers(0, len(pre5), (1000, len(pre5)))
    obs_ch = post5.absr.values[ia].mean(1) - pre5.absr.values[ib].mean(1)
    null_ch = post5.null_mean.values[ia].mean(1) - pre5.null_mean.values[ib].mean(1)
    row["obs_change"] = post5.absr.mean() - pre5.absr.mean(); row["null_change"] = post5.null_mean.mean() - pre5.null_mean.mean()
    row["a_ci_lo"] = np.percentile(obs_ch - null_ch, 2.5); row["a_ci_hi"] = np.percentile(obs_ch - null_ch, 97.5)
    row["b_null95"] = np.percentile(null_ch, 95)
    rows.append(row)
F5 = pd.DataFrame(rows)
print(F5.to_string(float_format=lambda v: f"{v:.3g}"))
ok_ep = F5[["gt_PRE", "gt_MAZE", "gt_POST"]].all(axis=1) & (F5[["p_PRE", "p_MAZE", "p_POST"]] < 0.05).all(axis=1)
ok_boot = (F5.a_ci_lo > 0) | (F5.obs_change > F5.b_null95)
print("variant, against each event's null mean: readings with p < 0.05 in all epochs", int((F5[["p_evmean_PRE", "p_evmean_MAZE", "p_evmean_POST"]] < 0.05).all(axis=1).sum()))
print("readings with observed > null in all epochs:", int(ok_ep.sum()), "of", len(F5), "; with the bootstrap reading (a):",
      int((F5.a_ci_lo > 0).sum()), "(b):", int((F5.obs_change > F5.b_null95).sum()), "; both parts:", int((ok_ep & ok_boot).sum()))
# seeds of the bootstrap, main reading
for k in [0, 1000, 2000, 3000, 4000]:
    rng = np.random.default_rng(seed("figs5b-abs-r-above-null") + k)
    x5 = sel(ST); pre5, post5 = x5[x5.epoch == "PRE"], x5[x5.epoch == "POST"]
    ia = rng.integers(0, len(post5), (1000, len(post5))); ib = rng.integers(0, len(pre5), (1000, len(pre5)))
    dd_ = (post5.absr.values[ia].mean(1) - pre5.absr.values[ib].mean(1)) - (post5.null_mean.values[ia].mean(1) - pre5.null_mean.values[ib].mean(1))
    print(f"seed +{k}: CI of observed - null change {np.percentile(dd_, [2.5, 97.5]).round(4)}")

**Result: reproduced.** Observed |r| exceeds the pooled null |r| (20 null samples kept per event) in PRE, MAZE and POST at rank-sum p below 1e-18 under all 24 readings, and the observed POST - PRE change of mean |r| exceeds the null change under both bootstrap readings in all 24 (main reading: 0.0149 against 0.0046, interval of the difference 0.0078 to 0.0125; the five seeds give lower bounds 0.0078 to 0.0080). Compared with each event's own null mean, PRE |r| is not higher (p = 1): the observed |r| are more spread than the null means.

## figs5e-duration-unchanged

> Ripple-event duration did not differ between PRE and POST (median 183 ms against 186 ms).

In [ ]:
rows = []
for rd in [(Y, L, S_) for Y in g.READINGS_Y for L in g.READINGS_L for S_ in g.READINGS_S]:
    xb = sel(ST, *rd).drop_duplicates(["session", "ev"])
    xa = ET[g.reading_mask(ET, *rd)]
    row = dict(zip("YLS", rd))
    for lab, z in [("replay", xb), ("all", xa)]:
        a, b = z[z.epoch == "PRE"].dur * 1000, z[z.epoch == "POST"].dur * 1000
        row[f"{lab}_PRE_med"] = a.median(); row[f"{lab}_POST_med"] = b.median(); row[f"{lab}_p"] = stats.ranksums(a, b).pvalue
        row[f"{lab}_n"] = (len(a), len(b))
    rows.append(row)
print(pd.DataFrame(rows).to_string(float_format=lambda v: f"{v:.3g}"))

**Result: not reproduced, resting on the yoking reading Y-a.** On the events of the replay analysis the PRE and POST medians are 180 and 176 ms, both within tolerance, but the paper's rank-sum test finds the difference under the main reading (p = 0.000531). Of the 12 readings of Y, L and S the test is significant under 8, and not under Y-a with L-b and S-a (p 0.0763), Y-b with L-a and S-a (0.455), Y-b with L-b and S-a (0.717) and Y-b with L-b and S-b (0.116). Under Y-b (the first T of each epoch) with L-a and S-a the medians (178 and 176 ms) are within tolerance and the test is not significant, which would be reproduced, so the outcome rests on Y-a. On all ripple events the difference is significant under every reading (medians 158 and 152 ms under the main one). The difference is 4 ms, 2% of the median.

## figs5f-more-participation-post

> A greater percentage of place cells participated (fired at least one spike) in POST than in PRE ripple events.

In [ ]:
rows = []
for rd in READ:
    x6 = sel(ST, *rd)
    pct = 100 * x6.n_active / x6.n_place
    a, b = pct[x6.epoch == "PRE"], pct[x6.epoch == "POST"]
    rows.append({**dict(zip("YLSD", rd)), "PRE_mean": a.mean(), "POST_mean": b.mean(), "PRE_med": a.median(), "POST_med": b.median(),
                 "p": stats.ranksums(b, a, alternative="greater").pvalue})
F6 = pd.DataFrame(rows)
print(F6.to_string(float_format=lambda v: f"{v:.3g}"))
print("readings with POST > PRE at p < 0.05:", int((F6.p < 0.05).sum()), "of", len(F6))

**Result: reproduced.** The percentage of place cells firing in an event is higher in POST than in PRE under all 24 readings (main reading: mean 34.9% in PRE against 35.6% in POST, rank-sum p 2.66e-08; largest p over readings below 0.05), a small difference.

## Per cell contribution

PCC per event as in the Methods (equation 8), with 100 cell-specific shuffles. A unit is a place cell in one running direction (its template), as in the paper's n = 491.

In [ ]:
PT = g.pcc_table(R, ST)
print("PCC rows (event, direction, cell):", len(PT))
CP = {}
for Y in g.READINGS_Y:
    for D in g.READINGS_D:
        CP[(Y, D)] = g.cell_pcc(PT, g.reading_mask(PT, Y, "L-a", "S-a", D))
C0 = CP[("Y-a", "D-a")]
print("units with PCC:", len(C0), "with both PRE and POST:", int(C0.dpcc.notna().sum()))
print(f"{time.time() - T0:.0f} s")

Shuffle counts. This notebook uses 200 null templates and 100 cell-specific shuffles where the paper uses 1,000 of each. On the smallest session the scores are recomputed with 1,000 of each, on the same place cells and events, and compared.

In [ ]:
# Shuffle counts (verifier round 1): Buddy_06272013, the smallest session, rescored with the paper's 1,000
# null templates per event and 1,000 cell-specific shuffles per cell and event, on the same place cells and events.
nm = "Buddy_06272013"
r, s = R[nm], S[nm]
rng_k = np.random.default_rng(seed("fig2b-strong-contributors"))
t1 = time.time()
krows = []
for d, dd in r["dirs"].items():
    cells = dd["place_cells"]; idx = [r["pyr"].index(u) for u in cells]
    tpl = g.Template(cells, dd["sm"][idx], dd["raw"][idx], rng_k, n_shuf=1000)
    spk = [s["spikes"][u] for u in cells]
    for sc in dd["scores"]:
        a, b = r["events"]["t"][sc["event"]]
        ep = r["events"]["epoch"][sc["event"]]
        o = g.score_event(tpl, g.bin_event(spk, a, b), pcc_k=1000 if ep in ("PRE", "POST") else 0)
        for u, p in zip(o["active"], o.get("pcc", [np.nan] * len(o["active"]))):
            krows.append((d, sc["event"], int(u), o["rz"], p))
print(f"{nm} rescored with 1,000 and 1,000 in {time.time() - t1:.0f} s")
K1 = pd.DataFrame(krows, columns=["dir", "ev", "cell", "rz1000", "pcc1000"])
P0 = PT[PT.session == nm]
M_ = P0.merge(K1, on=["dir", "ev", "cell"])
ev_ = M_.drop_duplicates(["dir", "ev"])
print(f"events {len(ev_)}: r(rZ with 200 nulls, rZ with 1,000) {np.corrcoef(ev_.rz, ev_.rz1000)[0, 1]:.4f}; "
      f"PCC rows {len(M_)}: r(PCC with 100 shuffles, with 1,000) {np.corrcoef(M_.pcc, M_.pcc1000)[0, 1]:.4f}")
for Y, D in [("Y-a", "D-a"), ("Y-a", "D-b")]:
    m = g.reading_mask(M_, Y, "L-a", "S-a", D)
    x = M_[m]
    pre, post = x[x.epoch == "PRE"].drop_duplicates(["dir", "ev"]), x[x.epoch == "POST"].drop_duplicates(["dir", "ev"])
    print(f"{Y} {D}: POST - PRE rZ {post.rz.mean() - pre.rz.mean():.4f} (200 nulls) and {post.rz1000.mean() - pre.rz1000.mean():.4f} (1,000)")
    A = g.cell_pcc(x, np.ones(len(x), bool))
    B = g.cell_pcc(x.assign(pcc=x.pcc1000), np.ones(len(x), bool))
    ka, kb = g.classify(A, "pa"), g.classify(B, "pa")
    j = ka.merge(kb, on=["session", "dir", "cell"], suffixes=("_100", "_1000"))
    print(f"   units {len(j)}: strongly contributing {int(j.strong_100.sum())} (100 shuffles) and {int(j.strong_1000.sum())} (1,000); "
          f"same label in {int((j.strong_100 == j.strong_1000).sum())}; PRE-significant {int(j.sig_PRE_100.sum())} and {int(j.sig_PRE_1000.sum())}, "
          f"POST-significant {int(j.sig_POST_100.sum())} and {int(j.sig_POST_1000.sum())}; r(dPCC) {np.corrcoef(j.dpcc_100, j.dpcc_1000)[0, 1]:.4f}; "
          f"rigid/plastic label the same in {int((j.group_100 == j.group_1000).sum())} of {len(j)}")

## figs9b-pcc-post-higher

> Place cells contribute more to POST than to PRE sequence content (median PCC -0.011 in PRE, 0.183 in POST), and the median change dPCC is 0.159.

In [ ]:
rows = []
for (Y, D), C in CP.items():
    rows.append({"Y": Y, "D": D, "n": len(C), "med_PRE": C.pcc_PRE.median(), "med_POST": C.pcc_POST.median(),
                 "med_dPCC": C.dpcc.median(), "p_ranksum": stats.ranksums(C.pcc_POST.dropna(), C.pcc_PRE.dropna()).pvalue,
                 "p_signrank": stats.wilcoxon(C.dpcc.dropna()).pvalue})
F9 = pd.DataFrame(rows)
print(F9.to_string(float_format=lambda v: f"{v:.3g}"))
exp_ = [(-0.011, 0.02), (0.183, 0.0183), (0.159, 0.0159)]
for _, rw in F9.iterrows():
    w = [abs(v - e) <= t for v, (e, t) in zip([rw.med_PRE, rw.med_POST, rw.med_dPCC], exp_)]
    print(rw.Y, rw.D, "within tolerance (PRE, POST, dPCC):", w)
for nm, gC in C0.groupby("session"):
    print(nm, f"median dPCC {gC.dpcc.median():.3f}, n {len(gC)}")

**Result: partially reproduced.** POST PCC exceeds PRE PCC (rank-sum p 3.38e-07) and dPCC exceeds zero (signed-rank p 6.4e-14) under the main reading, and both tests hold under all six readings of Y and D. The medians are outside tolerance under every reading: PRE 0.0185 (expected -0.011 within 0.02), POST 0.134 (0.183 within 0.0183), dPCC 0.134 (0.159 within 0.0159). By session the median dPCC ranges from -0.046 to 0.420.

## fig2b-strong-contributors, fig2-pre-post-overlap and rigid-plastic-counts

> 216 place cells contributed significantly to PRE or POST sequence content; 97 in PRE, 190 in POST, 71 in both, 26 in PRE only and 119 in POST only.

In [ ]:
rows = []
CL = {}
for (Y, D), C in CP.items():
    for P in ["pa", "pb"]:
        K = g.classify(C, P)
        CL[(Y, D, P)] = K
        npre, npost = int(K.sig_PRE.sum()), int(K.sig_POST.sum())
        nboth = int((K.sig_PRE & K.sig_POST).sum())
        rows.append({"Y": Y, "D": D, "P": P, "strong": int(K.strong.sum()), "PRE": npre, "POST": npost, "both": nboth,
                     "PRE_only": npre - nboth, "POST_only": npost - nboth, "overlap_%": 100 * nboth / max(npre, 1),
                     "chance_%": 100 * npost / len(K), "rigid": int((K.group == "rigid").sum()),
                     "plastic": int((K.group == "plastic").sum())})
F2b = pd.DataFrame(rows)
print(F2b.to_string(float_format=lambda v: f"{v:.3g}"))
K0 = CL[("Y-a", "D-a", "pa")]
print("main reading (Y-a, D-a, P-a) per session:")
print(K0.groupby("session").agg(units=("cell", "size"), strong=("strong", "sum"), rigid=("group", lambda v: (v == "rigid").sum()),
                                plastic=("group", lambda v: (v == "plastic").sum())).to_string())

**Result: not reproduced, resting on reading D-a.** Under the main reading D-a (each event scored with each direction's template, which the Methods' "Bayesian analysis was performed independently for the left and right direction of movement" describes) and the signed-rank test P-a the counts are 246 strongly contributing cells (216 expected within 21.6), 120 in PRE (97), 212 in POST (190) and 86 in both (71), all above tolerance. Under reading D-b (one direction per event) they are 217, 98, 172 and 53: the total and the PRE and POST counts are within tolerance and the overlap is not. The rank-sum reading P-b finds 13 to 24 cells under every reading, far from the paper. With the 5 cm kernel read as its full width at half maximum (cell `kernel-width-classification`, 507 place cells), D-a with P-a gives 232, 102, 200 and 70, all four within tolerance (D-b 200, 85, 164 and 49). The outcome therefore rests on the claims file's SD reading of the kernel as well as on D-a; the kernel reading is asked of a reader.

**Result: reproduced, resting on reading D-a.** Under the main reading (Y-a, D-a, P-a) 86 of 120 PRE-significant cells are also POST-significant, 71.7% (73 expected, within 7.3), against a chance level of 37%. Under D-b the overlap is 54.1% (Y-a), outside tolerance. With the kernel read as its full width the overlap is 68.6% under D-a, within tolerance, and 57.6% under D-b.

**Result: not reproduced, resting on reading D-a.** Under the main reading the split gives 120 rigid and 120 plastic cells (105 expected, within 10.5). Under reading D-b it gives 104 and 104 (Y-a), within tolerance, so the outcome turns on the direction reading; D-a is kept for every claim, as the Methods' independent analysis of each running direction suggests. With the kernel read as its full width, D-a gives 112 and 112, within tolerance (D-b 97 and 97), so the outcome rests on the claims file's SD reading of the kernel as well as on the direction reading.

### The kernel reading carried into the classification

The place-cell count turns on whether the 5 cm kernel is its SD (the claims file) or its full width at half maximum (cell of place-cell-count). Every claim from fig2b-strong-contributors on uses the place cells of the SD reading. Here the width reading is carried through: its place cells, rate maps and template shuffles (SD 2.12 cm), every PRE and POST candidate event rescored with the main analysis's event criteria and shuffle counts, then the PCC, the strongly contributing tests and the rigid and plastic split. It is a variant, shown for the counts of fig2b-strong-contributors, fig2-pre-post-overlap and rigid-plastic-counts; the later claims keep the SD reading.

In [ ]:
# Width reading of the 5 cm kernel (verifier round 2): place cells W_PC from the place-cell-count cell
rng_w = np.random.default_rng(seed("fig2b-strong-contributors", 2))
t1 = time.time()
wrows = []
for nm, r in R.items():
    s = S[nm]; pyr = r["pyr"]; evt = r["events"]["t"]; epc = r["events"]["epoch"]
    for d, dd in r["dirs"].items():
        cells = W_PC[(nm, d)]
        idx = [pyr.index(u) for u in cells]
        sm, raw = g.rate_maps(dd["occ"], dd["counts"][idx], sd_cm=SD_W)
        tpl = g.Template(cells, sm, raw, rng_w, n_shuf=g.N_NULL, sd_cm=SD_W)
        need = max(5, int(np.ceil(0.1 * len(cells))))
        spk = [s["spikes"][u] for u in cells]
        for i, (a, b) in enumerate(evt):
            if epc[i] not in ("PRE", "POST") or b - a < g.REPLAY_MIN_DUR:
                continue
            n = g.bin_event(spk, a, b)
            if (n.sum(1) > 0).sum() < need:
                continue
            o = g.score_event(tpl, n, pcc_k=g.N_PCC)
            for u, p, rc in zip(o["active"], o["pcc"], o["rz_cellshuf_mean"]):
                wrows.append((nm, d, i, int(u), p, o["rz"], rc))
print(f"place cells under the width reading {sum(len(v) for v in W_PC.values())}; rescored in {time.time() - t1:.0f} s")
PTw = pd.DataFrame(wrows, columns=["session", "dir", "ev", "cell", "pcc", "rz", "rz_cs"])
PTw = PTw.merge(ET[["session", "ev", "epoch", "Y-a", "Y-b", "Y-c", "L-a", "L-b", "S-a", "S-b", "dir_b"]], on=["session", "ev"], how="left")
PTw["D-a"] = True
PTw["D-b"] = (PTw["dir"] == PTw.dir_b) | (PTw["dir"] == "cw")
print("PCC rows (event, direction, cell):", len(PTw))
rows = []
CLw = {}
for Y in g.READINGS_Y:
    for D in g.READINGS_D:
        Cw = g.cell_pcc(PTw, g.reading_mask(PTw, Y, "L-a", "S-a", D))
        for P in ["pa", "pb"]:
            K = g.classify(Cw, P)
            CLw[(Y, D, P)] = K
            npre, npost = int(K.sig_PRE.sum()), int(K.sig_POST.sum())
            nboth = int((K.sig_PRE & K.sig_POST).sum())
            rows.append({"Y": Y, "D": D, "P": P, "units": len(K), "strong": int(K.strong.sum()), "PRE": npre, "POST": npost,
                         "both": nboth, "overlap_%": 100 * nboth / max(npre, 1), "chance_%": 100 * npost / len(K),
                         "rigid": int((K.group == "rigid").sum()), "plastic": int((K.group == "plastic").sum())})
F2w = pd.DataFrame(rows)
print("kernel read as its full width at half maximum:")
print(F2w.to_string(float_format=lambda v: f"{v:.3g}"))
print("within tolerance: strong 194.4 to 237.6, PRE 87.3 to 106.7, POST 171 to 209, both 63.9 to 78.1, overlap 65.7 to 80.3%, "
      "rigid and plastic 94.5 to 115.5")
Ka, Kw = CL[("Y-a", "D-a", "pa")], CLw[("Y-a", "D-a", "pa")]
j = Ka.merge(Kw, on=["session", "dir", "cell"], suffixes=("_sd", "_w"))
print(f"Y-a D-a P-a: units in both place-cell sets {len(j)}; strongly contributing in both {int((j.strong_sd & j.strong_w).sum())} "
      f"(SD reading {int(Ka.strong.sum())}, width reading {int(Kw.strong.sum())}); same rigid or plastic label "
      f"{int(((j.group_sd == j.group_w) & (j.group_sd != '')).sum())} of {int(((j.group_sd != '') & (j.group_w != '')).sum())} labelled in both")
print(f"{time.time() - T0:.0f} s")


## figs19c-pcc-change-per-cell

> Testing each place cell's per-event PCC scores between PRE and POST, 2.2% of cells had significantly higher PRE than POST scores and 21.4% significantly higher POST than PRE scores.

In [ ]:
rows = []
for (Y, D), C in CP.items():
    up = dn = 0
    for _, rw in C.iterrows():
        a, b = rw.vals_PRE, rw.vals_POST
        if len(a) >= 1 and len(b) >= 1:
            if stats.ranksums(b, a, alternative="greater").pvalue < 0.05: up += 1
            if stats.ranksums(a, b, alternative="greater").pvalue < 0.05: dn += 1
    rows.append({"Y": Y, "D": D, "units": len(C), "PRE>POST_%": 100 * dn / len(C), "POST>PRE_%": 100 * up / len(C),
                 "PRE>POST_%_of_491": 100 * dn / 491, "POST>PRE_%_of_491": 100 * up / 491})
print(pd.DataFrame(rows).to_string(float_format=lambda v: f"{v:.3g}"))

**Result: not reproduced.** Under the main reading 4.19% of units have PRE PCC significantly above POST and 15.7% POST above PRE (2.2% and 21.4% expected); with 491 as the denominator the values are 4.89% and 18.3%. No reading of Y or D puts either value within tolerance, though POST > PRE cells are three to four times as common as PRE > POST cells.

## Reruns of the replay analysis: fig2d-shuffle-high-dpcc, figs12-high-dpcc-exclusion, fig2e-remove-plastic

All six readings of Y and D (with L-a and S-a), PRE and POST events. The events of every session and direction are binned once and scored in batches (`rescore_batch`, helper). The baseline rerun is checked against the main scores: the same events, and rZ equal up to the null draw. Cells are ranked by dPCC within each session and direction under the same reading; the percentage is of the template's place cells. Null templates per event: 200 (Figure 2D and 2E use 200 and 100; the 120 exclusion reruns of Figure S12 use 50 each, for time).

In [ ]:
# Reruns of the replay analysis (Figures 2D, 2E, 4D, 4E and S12), batched. Every PRE and POST candidate
# event of L-a and S-a that falls in any yoked window is binned once per session and direction; each
# reading (Y, D) is a subset of these rows.
PREP = {}
for nm, r in R.items():
    s = S[nm]
    e = ET[(ET.session == nm) & ET["L-a"] & ET["S-a"] & (ET["Y-a"] | ET["Y-b"] | ET["Y-c"]) & ET.epoch.isin(["PRE", "POST"])]
    for d, dd in r["dirs"].items():
        pcs = dd["place_cells"]
        idx = [r["pyr"].index(u) for u in pcs]
        N_, st_, ln_, cnt_ = g.prepare_counts([s["spikes"][u] for u in pcs], e[["t0", "t1"]].values)
        PREP[(nm, d)] = {"cells": pcs, "sm": dd["sm"][idx], "raw": dd["raw"][idx], "N": N_, "st": st_, "ln": ln_,
                         "counts": cnt_, "dur": (e.t1 - e.t0).values, "ev": e.ev.values, "epoch": e.epoch.values,
                         **{Y: e[Y].values for Y in g.READINGS_Y}, "D-b": (e.dir_b.values == d) | (d == "cw")}
print("binned (event, direction) rows:", sum(len(p["ev"]) for p in PREP.values()))

def rows_of(p, Y, D):
    return p[Y] & (p["D-b"] if D == "D-b" else True)

def rerun_b(Y, D, keep_fn, rng, n_null, shuf_fn=None, n_obs=1, nulls=None):
    """Rerun under reading (Y, D) with L-a and S-a. Returns the scored rows, mean POST - PRE rZ, the PRE and
    POST rZ, and the POST - PRE difference for each shuffled template."""
    out, mats = [], []
    for (nm, d), p in PREP.items():
        pcs = p["cells"]
        ks = set(keep_fn(nm, d, pcs)); ss = set(shuf_fn(nm, d, pcs)) if shuf_fn else set()
        kidx = [i for i, u in enumerate(pcs) if u in ks]; sidx = [i for i, u in enumerate(pcs) if u in ss]
        ev, rz, rzm, mu, sd = g.rescore_batch(p, kidx, rng, n_null, sidx, n_obs, rows_of(p, Y, D),
                                              None if nulls is None else nulls[(nm, d)])
        out.append(pd.DataFrame({"session": nm, "dir": d, "row": ev, "ev": p["ev"][ev], "epoch": p["epoch"][ev], "rz": rz}))
        mats.append(rzm)
    df = pd.concat(out, ignore_index=True); M = np.vstack(mats)
    pre, post = (df.epoch == "PRE").values, (df.epoch == "POST").values
    per_tpl = M[post].mean(0) - M[pre].mean(0)
    return df, df.rz[post].mean() - df.rz[pre].mean(), df.rz[pre].values, df.rz[post].values, per_tpl

N_RERUN = 200     # null templates per event in the reruns (the paper: 1,000)
rng_null = np.random.default_rng(seed("fig2d-shuffle-high-dpcc"))
NULLS = {}
t1 = time.time()
for (nm, d), p in PREP.items():
    E = len(p["ev"])
    ev, rz, _, mu, sd = g.rescore_batch(p, range(len(p["cells"])), rng_null, N_RERUN)
    a, b = np.full(E, np.nan), np.full(E, np.nan); a[ev], b[ev] = mu, sd
    NULLS[(nm, d)] = (a, b)
print(f"null of the full templates, {N_RERUN} shuffles: {time.time() - t1:.0f} s")
# check: the baseline rerun selects the main analysis's events and reproduces its rZ up to the null draw
t1 = time.time()
BASE = {}
for Y in g.READINGS_Y:
    for D in g.READINGS_D:
        BASE[(Y, D)] = rerun_b(Y, D, lambda nm, d, p: p, None, N_RERUN, nulls=NULLS)
print(f"baselines {time.time() - t1:.0f} s")
for (Y, D), (df, dl, a, b, _) in BASE.items():
    ref = sel(ST, Y, "L-a", "S-a", D)
    ref = ref[ref.epoch.isin(["PRE", "POST"])]
    m = df.merge(ref[["session", "dir", "ev", "rz"]], on=["session", "dir", "ev"], how="outer", suffixes=("", "_main"), indicator=True)
    print(f"{Y} {D}: rerun rows {len(df)}, main rows {len(ref)}, in both {int((m._merge == 'both').sum())}; "
          f"r(rZ rerun, rZ main) {np.corrcoef(m.dropna().rz, m.dropna().rz_main)[0, 1]:.4f}; POST-PRE {dl:.4f} (main {ref[ref.epoch == 'POST'].rz.mean() - ref[ref.epoch == 'PRE'].rz.mean():.4f})")

In [ ]:
# fig2d-shuffle-high-dpcc: within each session and direction, the top (or bottom) pct% of the template's
# place cells by dPCC (ranked under the same reading) have their maps circularly shuffled; N_OBS shuffled
# templates per percentage, each event's rZ averaged over them, against the null of the intact template.
N_OBS = 100
rng_rr = np.random.default_rng(seed("fig2d-shuffle-high-dpcc", 1))
rng_b = np.random.default_rng(seed("fig2d-shuffle-high-dpcc", 2))
def ranked(dP, nm, d, pcs, high_first):
    have = [u for u in pcs if (nm, d, u) in dP]
    return sorted(have, key=lambda u: -dP[(nm, d, u)] if high_first else dP[(nm, d, u)])
def top(dP, nm, d, pcs, pct, high_first):
    return ranked(dP, nm, d, pcs, high_first)[:int(round(pct / 100 * len(pcs)))]
DPCC = {k: {(rw.session, rw.dir, rw.cell): rw.dpcc for rw in C.itertuples() if not np.isnan(rw.dpcc)} for k, C in CP.items()}
rows = []
t1 = time.time()
for (Y, D) in CP:
    dP = DPCC[(Y, D)]
    df0, d0, a0, b0, _ = BASE[(Y, D)]
    ci0 = g.boot_diff_ci(b0, a0, rng_b)
    rows.append({"Y": Y, "D": D, "from": "none", "pct": 0, "POST-PRE": d0, "ci_lo": ci0[0], "ci_hi": ci0[1], "tpl_lo": np.nan, "tpl_hi": np.nan,
                 "n_PRE": len(a0), "n_POST": len(b0)})
    for hf, pcts in [(True, [5, 10, 15, 20, 25]), (False, [50, 75])]:
        for pct in pcts:
            df, dl, a, b, pt = rerun_b(Y, D, lambda nm, d, p: p, rng_rr, N_RERUN,
                                       shuf_fn=lambda nm, d, p, pct=pct, hf=hf: top(dP, nm, d, p, pct, hf), n_obs=N_OBS, nulls=NULLS)
            ci = g.boot_diff_ci(b, a, rng_b)
            rows.append({"Y": Y, "D": D, "from": "highest" if hf else "lowest", "pct": pct, "POST-PRE": dl, "ci_lo": ci[0], "ci_hi": ci[1],
                         "tpl_lo": np.percentile(pt, 2.5), "tpl_hi": np.percentile(pt, 97.5), "n_PRE": len(a), "n_POST": len(b)})
F2d = pd.DataFrame(rows)
print(F2d.to_string(float_format=lambda v: f"{v:.3g}"))
print(f"fig2d reruns {time.time() - t1:.0f} s")
def first_elim(F, how):
    out = {}
    for (Y, D), gx in F[F["from"] == "highest"].groupby(["Y", "D"]):
        gx = gx.sort_values("pct")
        hit = gx[gx["POST-PRE"] <= 0] if how == "a" else gx[gx.ci_lo <= 0]
        out[(Y, D)] = int(hit.pct.iloc[0]) if len(hit) else None
    return out
for how in "ab":
    print(f"first eliminated (reading {how}):", first_elim(F2d, how))
low75 = F2d[(F2d["from"] == "lowest") & (F2d.pct == 75)]
print("shuffling the lowest 75%: interval above zero under", int((low75.ci_lo > 0).sum()), "of", len(low75), "readings")

**Result: reproduced, resting on the yoking reading Y-a and on averaging over shuffled templates.** With 100 shuffled templates per percentage, each event's rZ averaged over them, POST - PRE rZ under the main reading (Y-a, D-a) is 0.0772 with no cell shuffled, 0.00793 with the top 5% shuffled (interval -0.0117 to 0.0267) and -0.0227 with the top 10% (interval -0.0406 to -0.00463), below zero to 25%. Replay is first eliminated (reading a, mean at or below zero) at 10% under five of the six readings of Y and D, and at 5% under Y-c with D-b (-0.00253); under reading b (interval includes zero) it is first eliminated at 5% under all six. Shuffling the lowest 75% leaves the interval above zero under all six (main reading 0.0665, interval 0.0574 to 0.0757). A single shuffled template, as in the first round, can fall either side of zero at 5%: the 95% range of the per-template POST - PRE at 5% is -0.0113 to 0.023. The pass rests on reading (a) of "eliminated" as well as on Y-a: under reading (a), Y-c with D-b eliminates replay at 5%, below 9%, and under reading (b) replay is eliminated at 5% under all six readings.

In [ ]:
# figs12-high-dpcc-exclusion: cells excluded from the template and from the event criteria, each rerun
# with its own null (N_RERUN_X shuffles of the kept cells), all six readings of Y and D, 5% steps.
N_RERUN_X = 50    # 120 reruns with their own nulls; fewer shuffles for time
rng_rr = np.random.default_rng(seed("figs12-high-dpcc-exclusion"))
rng_b = np.random.default_rng(seed("figs12-high-dpcc-exclusion", 1))
rows = []
t1 = time.time()
for (Y, D) in CP:
    dP = DPCC[(Y, D)]
    df0, d0, a0, b0, _ = BASE[(Y, D)]
    ci0 = g.boot_diff_ci(b0, a0, rng_b)
    rows.append({"Y": Y, "D": D, "from": "none", "pct": 0, "POST-PRE": d0, "ci_lo": ci0[0], "ci_hi": ci0[1], "n_PRE": len(a0), "n_POST": len(b0)})
    for hf, pcts in [(True, [5, 10, 15, 20, 25]), (False, list(range(5, 80, 5)))]:
        for pct in pcts:
            df, dl, a, b, _ = rerun_b(Y, D, lambda nm, d, p, pct=pct, hf=hf: [u for u in p if u not in set(top(dP, nm, d, p, pct, hf))],
                                      rng_rr, N_RERUN_X)
            ci = g.boot_diff_ci(b, a, rng_b) if len(a) and len(b) else (np.nan, np.nan)
            rows.append({"Y": Y, "D": D, "from": "highest" if hf else "lowest", "pct": pct, "POST-PRE": dl, "ci_lo": ci[0], "ci_hi": ci[1],
                         "n_PRE": len(a), "n_POST": len(b)})
F12 = pd.DataFrame(rows)
print(F12[F12["from"] != "lowest"].to_string(float_format=lambda v: f"{v:.3g}"))
print(F12[F12["from"] == "lowest"].pivot_table(index="pct", columns=["Y", "D"], values="ci_lo").to_string(float_format=lambda v: f"{v:.3f}"))
print("lowest-dPCC exclusion, POST - PRE (rows pct, columns reading):")
print(F12[F12["from"] == "lowest"].pivot_table(index="pct", columns=["Y", "D"], values="POST-PRE").to_string(float_format=lambda v: f"{v:.3f}"))
print(f"figs12 reruns {time.time() - t1:.0f} s")
for (Y, D), gx in F12.groupby(["Y", "D"]):
    hi_ = gx[gx["from"] == "highest"].sort_values("pct"); lo_ = gx[gx["from"] == "lowest"]
    neg = hi_[hi_["POST-PRE"] < 0]
    print(f"{Y} {D}: first below zero at {int(neg.pct.iloc[0]) if len(neg) else None}%; lowest excluded, lower bound above zero at every step to 75%: "
          f"{bool((lo_.ci_lo > 0).all())} (smallest lower bound {lo_.ci_lo.min():.3f} at {int(lo_.loc[lo_.ci_lo.idxmin(), 'pct'])}%)")

**Result: reproduced.** Excluding the highest-dPCC cells, POST - PRE rZ first falls below zero at 10% under all six readings of Y and D (main reading: 0.023 at 5%, -0.022 at 10%, -0.068 at 25%). Excluding the lowest-dPCC cells in 5% steps, the lower bound of the interval stays above zero at every step to 75% under all six readings (smallest lower bound 0.075, at 5% under Y-c with D-a; main reading 0.17 at 75%). The replay reruns give values that differ in the third significant digit between executions of the same notebook, so the values that do are given here to two significant digits.

In [ ]:
# fig2e-remove-plastic: all rigid or all plastic cells (classification P-a under the same Y and D reading) removed
rng_rr = np.random.default_rng(seed("fig2e-remove-plastic"))
rng_b = np.random.default_rng(seed("fig2e-remove-plastic", 1))
rows = []
for (Y, D) in CP:
    K_ = CL[(Y, D, "pa")]
    grp = {(rw.session, rw.dir, rw.cell): rw.group for rw in K_.itertuples() if rw.group}
    df0, d0, a0, b0, _ = BASE[(Y, D)]
    ci0 = g.boot_diff_ci(b0, a0, rng_b)
    rows.append({"Y": Y, "D": D, "removed": "none", "n_removed": 0, "POST-PRE": d0, "ci_lo": ci0[0], "ci_hi": ci0[1]})
    for gname in ["rigid", "plastic"]:
        df, dl, a, b, _ = rerun_b(Y, D, lambda nm, d, p, gn=gname: [u for u in p if grp.get((nm, d, u)) != gn], rng_rr, 100)
        ci = g.boot_diff_ci(b, a, rng_b)
        rows.append({"Y": Y, "D": D, "removed": gname, "n_removed": sum(v == gname for v in grp.values()), "POST-PRE": dl,
                     "ci_lo": ci[0], "ci_hi": ci[1], "n_PRE": len(a), "n_POST": len(b)})
F2e = pd.DataFrame(rows)
print(F2e.to_string(float_format=lambda v: f"{v:.3g}"))
ok = [(gx[gx.removed == "plastic"].ci_lo.iloc[0] <= 0) and (gx[gx.removed == "rigid"].ci_lo.iloc[0] > 0) for _, gx in F2e.groupby(["Y", "D"])]
print("readings where removing plastic cells brings the interval to zero or below and removing rigid cells does not:", sum(ok), "of", len(ok))

**Result: reproduced.** Removing all plastic cells brings the interval of POST - PRE rZ below zero and removing all rigid cells leaves it above zero under all six readings of Y and D (main reading: plastic removed -0.032, interval -0.054 to -0.0096 or -0.0097; rigid removed 0.10, interval 0.078 to 0.12; with D-b groups -0.057 and 0.14). The replay reruns give values that differ in the third significant digit between executions of the same notebook, so the values that do are given here to two significant digits.

## Firing rates and ripple measures per cell

Offline time is the S-a offline state within the yoked windows (Y-a) for PRE and POST, and immobility on the maze for MAZE. Ripple events are all ripple events of the main reading (Y-a, L-a, S-a).

In [ ]:
cellrows = []
for nm, r in R.items():
    s = S[nm]
    off, yk = r["off"], r["yoke"]
    offY = {ep: g.intersect_intervals(off["S-a"][ep], yk["Y-a"][ep]) for ep in ["PRE", "POST"]}
    offY["MAZE"] = off["S-a"]["MAZE"]
    e = ET[(ET.session == nm) & g.reading_mask(ET, "Y-a", "L-a", "S-a")]
    evs = {ep: e[e.epoch == ep][["t0", "t1"]].values for ep in ["PRE", "MAZE", "POST"]}
    t = s["pos_t"]; dt = np.median(np.diff(t))
    runm = (r["speed"] >= g.RUN_SPEED) & np.isfinite(r["speed"])
    E = s["epochs"]
    for u in r["pyr"]:
        sp = s["spikes"][u]
        row = {"session": nm, "cell": u, "shank": u // 100}
        for ep in ["PRE", "MAZE", "POST"]:
            T_off = g.total(offY[ep])
            row[f"rate_{ep}"] = g.spikes_in(sp, offY[ep]) / T_off if T_off else np.nan
            Tev = g.total(evs[ep])
            row[f"riprate_{ep}"] = g.spikes_in(sp, evs[ep]) / Tev if Tev else np.nan
            row[f"gain_{ep}"] = row[f"riprate_{ep}"] / row[f"rate_{ep}"] if row[f"rate_{ep}"] > 0 else np.nan
            ia = g.concat_isis(sp, evs[ep], True); ib = g.concat_isis(sp, evs[ep], False)
            row[f"burst_a_{ep}"] = 100 * np.mean(ia < 0.006) if len(ia) else np.nan
            row[f"burst_b_{ep}"] = 100 * np.mean(ib < 0.006) if len(ib) else np.nan
        row["n_ev_PRE"], row["n_ev_POST"] = len(evs["PRE"]), len(evs["POST"])
        row["rate_MAZE_all"] = g.spikes_in(sp, E["MazeEpoch"]) / np.diff(E["MazeEpoch"][0])[0]
        nrun = np.sum(runm); idx = np.searchsorted(t, sp); idx = np.clip(idx, 0, len(t) - 1)
        inm = (sp >= t[0]) & (sp <= t[-1]) & (np.abs(t[idx] - sp) < dt)
        row["rate_MAZE_run"] = np.sum(runm[idx[inm]]) / (nrun * dt)
        T_all_off = g.total(offY["PRE"]) + g.total(offY["POST"])
        row["sess_a"] = (g.spikes_in(sp, offY["PRE"]) + g.spikes_in(sp, offY["POST"]) + g.spikes_in(sp, offY["MAZE"])) / (T_all_off + g.total(offY["MAZE"]))
        row["sess_b"] = (g.spikes_in(sp, offY["PRE"]) + g.spikes_in(sp, offY["POST"])) / T_all_off
        row["sess_c"] = len(sp) / s["duration"]
        evall = np.vstack([evs["PRE"], evs["MAZE"], evs["POST"]])
        evpp = np.vstack([evs["PRE"], evs["POST"]])
        rr_a = g.spikes_in(sp, evall) / g.total(evall); rr_b = g.spikes_in(sp, evpp) / g.total(evpp)
        row["sessgain_a"] = rr_a / row["sess_a"] if row["sess_a"] > 0 else np.nan
        row["sessgain_b"] = rr_b / row["sess_b"] if row["sess_b"] > 0 else np.nan
        row["sessgain_c"] = rr_a / row["sess_c"] if row["sess_c"] > 0 else np.nan
        row["rate_PRE_only"] = row["rate_PRE"]
        cellrows.append(row)
CELL = pd.DataFrame(cellrows).set_index(["session", "cell"])
CELL["dgain"] = CELL.gain_POST - CELL.gain_PRE
CELL["dburst_a"] = CELL.burst_a_POST - CELL.burst_a_PRE
CELL["dburst_b"] = CELL.burst_b_POST - CELL.burst_b_PRE
print(CELL.describe().T[["count", "mean", "50%", "min", "max"]].round(3).to_string())
# rigid and plastic units under the main reading D-a and under D-b (verifier round 1); `use` selects one
K0b = CL[("Y-a", "D-b", "pa")]
KS = {"D-a": K0, "D-b": K0b}
GRP = {DL: K_.join(CELL, on=["session", "cell"]) for DL, K_ in KS.items()}
def use(DL):
    global UNITS, RIG, PLA
    UNITS = GRP[DL]; RIG, PLA = UNITS[UNITS.group == "rigid"], UNITS[UNITS.group == "plastic"]
for DL in KS:
    use(DL); print(DL, "rigid units", len(RIG), "plastic units", len(PLA))
use("D-a")
def cmp(col, a=None, b=None):
    a = RIG if a is None else a; b = PLA if b is None else b
    x1, x2 = a[col].dropna(), b[col].dropna()
    return x1.median(), x2.median(), stats.ranksums(x1, x2).pvalue, len(x1), len(x2)

## fig4a-rigid-faster

> Rigid cells fire faster than plastic cells in offline states (session-wide median 0.76 against 0.54 Hz; PRE 0.762 against 0.479 Hz; POST 0.753 against 0.472 Hz), but not during maze running (1.09 against 1.13 Hz).

In [ ]:
for DL in KS:
    use(DL); print(f"-- groups of reading {DL}")
    for col in ["sess_a", "sess_b", "sess_c", "rate_PRE", "rate_POST", "rate_MAZE_all", "rate_MAZE_run"]:
        m1, m2, p, n1, n2 = cmp(col)
        print(f"{col:15s} rigid median {m1:.3f} Hz, plastic {m2:.3f} Hz, rank-sum p {p:.3g} (n {n1}, {n2})")
use("D-a")
for P in ["pb"]:
    Kb = CL[("Y-a", "D-a", "pb")].join(CELL, on=["session", "cell"])
    for col in ["sess_b", "rate_PRE", "rate_POST", "rate_MAZE_run"]:
        x1, x2 = Kb[Kb.group == "rigid"][col].dropna(), Kb[Kb.group == "plastic"][col].dropna()
        print(f"variant P-b {col:12s} rigid {x1.median():.3f}, plastic {x2.median():.3f}, p {stats.ranksums(x1, x2).pvalue:.3g}")
print(UNITS[UNITS.group != ""].groupby(["session", "group"]).sess_b.median().unstack().round(3).to_string())

**Result: not reproduced.** With the groups of the main reading D-a rigid cells fire faster than plastic cells offline session-wide (reading a 0.713 against 0.527 Hz, p 0.0412; reading b 0.693 against 0.520 Hz, p 0.0278) and in POST (0.659 against 0.472 Hz, p 0.0162), but the PRE difference (0.652 against 0.532 Hz, p 0.0659) and reading c (p 0.108) are not significant, and the criterion asks for the session-wide, PRE and POST tests together. The MAZE rates do not differ (all MAZE time p 0.159, running p 0.117). With the groups of D-b none of the offline differences is significant (session-wide p 0.235 to 0.315, PRE p 0.261, POST p 0.263).

## figs14-plastic-ripple-gain and figs14-plastic-ripple-bursting

> Plastic cells have a higher within-ripple firing rate gain than rigid cells (POST median 5.75 against 4.52; PRE 4.91 against 4.3; session-wide 5.03 against 4.57) and a larger PRE to POST increase in gain (0.81 against 0.14).
> Plastic cells burst more within ripples than rigid cells in POST (median 27.3% against 23.3% of ISIs under 6 ms) and increase their within-ripple bursting more from PRE to POST (2.81 against 1.13 percentage points).

In [ ]:
for DL in KS:
    use(DL); print(f"-- groups of reading {DL}")
    for col in ["gain_PRE", "gain_POST", "gain_MAZE", "sessgain_a", "sessgain_b", "sessgain_c", "dgain",
                "burst_a_POST", "burst_b_POST", "dburst_a", "dburst_b", "burst_a_PRE", "burst_b_PRE"]:
        m1, m2, p, n1, n2 = cmp(col)
        print(f"{col:13s} rigid median {m1:.3f}, plastic {m2:.3f}, rank-sum p {p:.3g} (n {n1}, {n2})")
use("D-a")

**Result: not reproduced.** With the D-a groups plastic cells have a higher within-ripple gain in POST (5.763 against 4.757, p 0.00277), session-wide (reading a 4.833 against 4.518, p 0.0234) and a larger change (0.790 against 0.259, p 0.0311), but the PRE difference (4.754 against 4.528, p 0.173) is not significant, and the criterion asks for all four tests. With the D-b groups only the POST difference is significant (p 0.0106; PRE p 0.303, session-wide p 0.0596 to 0.136, change p 0.0511).

**Result: not reproduced.** POST within-ripple burstiness does not differ between groups (D-a: reading a plastic 26.267% against rigid 24.062%, p 0.264; reading b p 0.233), and the PRE to POST change is not larger in plastic cells (reading a 1.581 against 2.403 points, p 0.2). The POST medians of reading a are within tolerance. With the D-b groups no test is significant either (p 0.143 to 0.769).

## figs16ab-plastic-synchrony-increase

> The PRE to POST increase in ripple-triggered firing is present in plastic but not rigid cells, and the PRE to POST increase in pair-wise coactivity is larger among plastic cells.

In [ ]:
# pair-wise coactivity, 100 ms bins (Methods, Pair-Wise Co-Activity Analysis)
pairrows = []
VEC = {}
for nm, r in R.items():
    s = S[nm]
    off, yk = r["off"], r["yoke"]
    ivs = {ep: g.intersect_intervals(off["S-a"][ep], yk["Y-a"][ep]) for ep in ["PRE", "POST"]}
    t = s["pos_t"]; dt = np.median(np.diff(t))
    runm = (r["speed"] >= g.RUN_SPEED) & np.isfinite(r["speed"])
    lab_, nl_ = __import__("scipy").ndimage.label(runm)
    objs = __import__("scipy").ndimage.find_objects(lab_)
    ivs["RUN"] = np.array([(t[o[0].start], t[o[0].stop - 1] + dt) for o in objs]).reshape(-1, 2)
    pcs = sorted(set().union(*[set(dd["place_cells"]) for dd in r["dirs"].values()]))
    V = {ep: np.array([g.binned_counts(s["spikes"][u], ivs[ep], 0.1) for u in pcs], float) for ep in ivs}
    CC = {ep: np.corrcoef(V[ep]) for ep in V}
    VEC[nm] = (pcs, ivs)
    for i in range(len(pcs)):
        for j in range(i + 1, len(pcs)):
            if pcs[i] // 100 == pcs[j] // 100:
                continue
            pairrows.append({"session": nm, "c1": pcs[i], "c2": pcs[j], **{ep: CC[ep][i, j] for ep in CC}})
PAIRS = pd.DataFrame(pairrows).dropna()
PAIRS["dco"] = PAIRS.POST - PAIRS.PRE
print("pairs on different shanks:", len(PAIRS), "per session:", PAIRS.groupby("session").size().to_dict())
CG = {}
for DL, K_ in KS.items():
    cg = {}
    for rw in K_[K_.group != ""].itertuples():
        cg.setdefault((rw.session, rw.cell), set()).add(rw.group)
    CG[DL] = {k: next(iter(v)) for k, v in cg.items() if len(v) == 1}
    print(f"{DL}: cells rigid in every direction they are classified:", sum(v == "rigid" for v in CG[DL].values()),
          "plastic:", sum(v == "plastic" for v in CG[DL].values()), "both labels (dropped from pair groups):", sum(len(v) > 1 for v in cg.values()))
    PAIRS[f"g1_{DL}"] = [CG[DL].get((a, b), "") for a, b in zip(PAIRS.session, PAIRS.c1)]
    PAIRS[f"g2_{DL}"] = [CG[DL].get((a, b), "") for a, b in zip(PAIRS.session, PAIRS.c2)]
cgroup = CG["D-a"]
PAIRS["g1"], PAIRS["g2"] = PAIRS["g1_D-a"], PAIRS["g2_D-a"]
print(f"{time.time() - T0:.0f} s")

In [ ]:
# (A) ripple-triggered rate, -50 to 100 ms around event onset, 10 ms bins
edges = np.arange(-0.05, 0.1001, 0.01)
trig = {}
for nm, r in R.items():
    s = S[nm]
    e = ET[(ET.session == nm) & g.reading_mask(ET, "Y-a", "L-a", "S-a")]
    for u in r["pyr"]:
        sp = s["spikes"][u]
        for ep in ["PRE", "POST"]:
            on = e[e.epoch == ep].t0.values
            h = np.zeros(len(edges) - 1)
            lo = np.searchsorted(sp, on - 0.05); hi = np.searchsorted(sp, on + 0.1)
            rel = np.concatenate([sp[a:b] - o for a, b, o in zip(lo, hi, on)]) if len(on) else np.zeros(0)
            h = np.histogram(rel, edges)[0] / (len(on) * 0.01) if len(on) else h
            trig[(nm, u, ep)] = h
for DL in KS:
  use(DL); print(f"-- groups of reading {DL}")
  for gname, G_ in [("rigid", RIG), ("plastic", PLA), ("all units", UNITS)]:
      d_a = np.array([trig[(rw.session, rw.cell, "POST")][5:].mean() - trig[(rw.session, rw.cell, "PRE")][5:].mean() for rw in G_.itertuples()])
      mean_curve = np.mean([trig[(rw.session, rw.cell, "PRE")] + trig[(rw.session, rw.cell, "POST")] for rw in G_.itertuples()], 0)
      pk = int(np.argmax(mean_curve))
      d_b = np.array([trig[(rw.session, rw.cell, "POST")][pk] - trig[(rw.session, rw.cell, "PRE")][pk] for rw in G_.itertuples()])
      print(f"{gname}: (a) POST-PRE mean rate 0-100 ms median {np.median(d_a):.3f} Hz, signed-rank p {stats.wilcoxon(d_a).pvalue:.3g}; "
            f"(b) at peak bin {edges[pk]*1000:.0f} ms median {np.median(d_b):.3f} Hz, p {stats.wilcoxon(d_b).pvalue:.3g} (n {len(d_a)})")
# (B) change in coactivity per unit
def per_cell_change(nm, c, own=None, DL="D-a"):
    m = (PAIRS.session == nm) & ((PAIRS.c1 == c) | (PAIRS.c2 == c))
    x = PAIRS[m]
    if own:
        other = np.where(x.c1 == c, x[f"g2_{DL}"], x[f"g1_{DL}"])
        x = x[other == own]
    return x.dco.mean() if len(x) else np.nan
for DL in KS:
    use(DL); print(f"-- groups of reading {DL}")
    for lab, own in [("(c) all pairs", False), ("(d) own group", True)]:
        v1 = [per_cell_change(rw.session, rw.cell, "rigid" if own else None, DL) for rw in RIG.itertuples()]
        v2 = [per_cell_change(rw.session, rw.cell, "plastic" if own else None, DL) for rw in PLA.itertuples()]
        v1, v2 = np.array(v1)[~np.isnan(v1)], np.array(v2)[~np.isnan(v2)]
        print(f"{lab}: coactivity change rigid median {np.median(v1):.4f}, plastic {np.median(v2):.4f}, plastic > rigid rank-sum p "
              f"{stats.ranksums(v2, v1, alternative='greater').pvalue:.3g} (n {len(v1)}, {len(v2)})")
    for gname, mm in [("rigid-rigid", (PAIRS[f"g1_{DL}"] == "rigid") & (PAIRS[f"g2_{DL}"] == "rigid")),
                      ("plastic-plastic", (PAIRS[f"g1_{DL}"] == "plastic") & (PAIRS[f"g2_{DL}"] == "plastic"))]:
        x = PAIRS[mm]
        print(f"{gname} pairs: n {len(x)}, PRE mean {x.PRE.mean():.4f}, POST mean {x.POST.mean():.4f}, signed-rank p {stats.wilcoxon(x.dco).pvalue:.3g}")
use("D-a")

**Result: not reproduced, resting on reading D-a.** With the D-a groups the change in pair-wise coactivity is larger in plastic than in rigid cells (reading c: median 0.0080 against 0.0054, p 0.0398; reading d: 0.0089 against 0.0039, p 0.00942), but the ripple-triggered rate increase from PRE to POST is not significant in plastic cells under either reading (a: p 0.311; b: p 0.0793), and the criterion asks for it. With the D-b groups the claim holds: the plastic increase is significant at the peak bin (reading b, +0.405 Hz, p 0.0157) and the rigid one is not (p 0.486), and the coactivity change is larger in plastic cells (p 0.0229 and 0.000469).

## rate-split-medians

> Splitting place cells into equal halves by offline firing rate gives low-rate cells with median 0.39 Hz and high-rate cells with median 1.12 Hz.

The units are the place cells per running direction (n as in place-cell-count).

In [ ]:
PCU = pd.DataFrame([{"session": nm, "dir": d, "cell": u} for nm, r in R.items() for d, dd in r["dirs"].items() for u in dd["place_cells"]])
PCU = PCU.join(CELL, on=["session", "cell"])
PCU["rate_i"] = PCU.sess_b
PCU["rate_ii"] = PCU.rate_PRE
def split(df, col, how):
    lab = pd.Series("", index=df.index)
    keys = {"a": ["session", "dir"], "b": ["session"], "c": None}[how]
    groups = df.groupby(keys) if keys else [(None, df)]
    for _, gx in groups:
        gx = gx.sort_values(col)
        n = len(gx); idx = list(gx.index)
        if n % 2 == 1:
            idx.pop(n // 2)
        lab[idx[:len(idx) // 2]] = "low"; lab[idx[len(idx) // 2:]] = "high"
    return lab
rows = []
for how in "abc":
    for col in ["rate_i", "rate_ii"]:
        lab = split(PCU, col, how)
        lo, hi = PCU[lab == "low"][col], PCU[lab == "high"][col]
        rows.append({"split": how, "rate": col, "low_med": lo.median(), "high_med": hi.median(), "n_low": len(lo), "n_high": len(hi),
                     "within": abs(lo.median() - 0.39) <= 0.039 and abs(hi.median() - 1.12) <= 0.112})
FR = pd.DataFrame(rows)
print(FR.to_string(float_format=lambda v: f"{v:.3f}"))
match = FR[FR.within]
SPLIT = (match.iloc[0].split, match.iloc[0].rate) if len(match) else ("a", "rate_i")
print("split used by the dependent claims:", SPLIT, "(matched)" if len(match) else "(none matched; reading (a)(i) used)")
PCU["rgroup"] = split(PCU, SPLIT[1], SPLIT[0])

**Result: reproduced.** Every split reading gives medians within tolerance; the first, a split within session and direction on the offline PRE and POST rate, gives 0.392 and 1.104 Hz (0.39 and 1.12 expected).

## figs16c-low-rate-coding

> Low-rate cells carry more spatial information per spike (0.91 against 0.56 bits/spike), have a higher in/out place-field rate ratio (6.38 against 3.86), fewer place fields (log-mean 1.29 against 1.59), and a larger PRE to POST increase in within-ripple gain (0.71 against 0.06) than high-rate cells.

In [ ]:
# place-field measures per unit (session, direction, cell), as in reference 32
mrows = []
LAPS = {}
for nm, r in R.items():
    s = S[nm]
    LAPS[nm] = g.laps(s, r["speed"], r["v1"])
    for d, dd in r["dirs"].items():
        nb = dd["occ"].shape[0]
        t = s["pos_t"]; dt = np.median(np.diff(t))
        okm = g.run_mask(s, d, r["speed"], r["v1"])
        tmid = np.median(t[okm]) if okm.any() else t.mean()
        for u in dd["place_cells"]:
            i = r["pyr"].index(u)
            sm, raw, occ = dd["sm"][i], dd["raw"][i], dd["occ"]
            fl = dd["fields"][u]
            infield = np.zeros(nb, bool)
            for a, c in fl: infield[a:c] = True
            v = occ > 0
            rin = dd["counts"][i][infield].sum() / occ[infield].sum()
            rout = dd["counts"][i][~infield & v].sum() / max(occ[~infield & v].sum(), 1e-9)
            ips, ipsec = g.spatial_info(sm, occ)
            coh = g.spatial_coherence(raw, occ)
            # stability (a) first vs second half of the running samples; (b) odd vs even laps
            def map_for(mask):
                b = np.clip(np.floor(np.nan_to_num(s["pos_1d"]) / (g.BIN_CM / 100)).astype(int), 0, nb - 1)
                o = np.bincount(b[mask], minlength=nb) * dt
                sp = s["spikes"][u]; k = np.clip(np.searchsorted(t, sp), 1, len(t) - 1)
                k = np.where(np.abs(t[k - 1] - sp) < np.abs(t[k] - sp), k - 1, k)
                kk = k[mask[k] & (np.abs(t[k] - sp) < dt)]
                cnt = np.bincount(b[kk], minlength=nb)
                so = g.smooth(o)
                with np.errstate(invalid="ignore", divide="ignore"):
                    return np.where(so > 0, g.smooth(cnt.astype(float)) / so, np.nan)
            m1, m2 = map_for(okm & (t < tmid)), map_for(okm & (t >= tmid))
            ok_ = ~np.isnan(m1) & ~np.isnan(m2)
            stab_a = np.corrcoef(m1[ok_], m2[ok_])[0, 1] if ok_.sum() > 3 else np.nan
            lp = [(a, c) for dd_, a, c in LAPS[nm] if dd_ == d]
            odd = np.zeros(len(t), bool); even = np.zeros(len(t), bool)
            for j, (a, c) in enumerate(lp):
                (odd if j % 2 == 0 else even)[(t >= a) & (t <= c)] = True
            m1, m2 = map_for(okm & odd), map_for(okm & even)
            ok_ = ~np.isnan(m1) & ~np.isnan(m2)
            stab_b = np.corrcoef(m1[ok_], m2[ok_])[0, 1] if ok_.sum() > 3 else np.nan
            mrows.append({"session": nm, "dir": d, "cell": u, "inout": rin / rout if rout > 0 else np.nan, "info_spike": ips,
                          "info_sec": ipsec, "coherence": coh, "stab_a": stab_a, "stab_b": stab_b, "n_fields": len(fl),
                          "peak_frac": (np.argmax(sm) + 0.5) / nb if d != "dec" else 1 - (np.argmax(sm) + 0.5) / nb})
PFM = pd.DataFrame(mrows).set_index(["session", "dir", "cell"])
GRP = {DL: U.join(PFM, on=["session", "dir", "cell"]) for DL, U in GRP.items()}
use("D-a")
PCU = PCU.join(PFM, on=["session", "dir", "cell"])
print(PFM.describe().T[["count", "mean", "50%", "min", "max"]].round(3).to_string())
print("laps per session:", {nm: len(v) for nm, v in LAPS.items()}, "NWB trials:", {nm: len(NWB[nm]["trials"][0]) if "trials" in NWB[nm] else 0 for nm in R})
print(f"{time.time() - T0:.0f} s")

In [ ]:
PCU["dgain"] = PCU.gain_POST - PCU.gain_PRE
lo, hi = PCU[PCU.rgroup == "low"], PCU[PCU.rgroup == "high"]
gm = lambda v: float(np.exp(np.mean(np.log(v))))
for col in ["info_spike", "inout", "n_fields", "dgain"]:
    a, b = lo[col].replace([np.inf, -np.inf], np.nan).dropna(), hi[col].replace([np.inf, -np.inf], np.nan).dropna()
    extra = f", geometric means {gm(a):.3f} and {gm(b):.3f}" if col == "n_fields" else ""
    print(f"{col:10s} low-rate median {a.median():.3f}, high-rate {b.median():.3f}{extra}, rank-sum p {stats.ranksums(a, b).pvalue:.3g} (n {len(a)}, {len(b)})")

**Result: partially reproduced.** All four rank-sum tests hold in the stated direction: information per spike 0.811 against 0.457 bits/spike (p 2.03e-15), in/out ratio 6.954 against 4.176 (p 1.13e-14), geometric mean number of fields 1.237 against 1.425 (p 0.00061), change in gain 0.810 against 0.210 (p 0.00326). The in/out ratios and the low-rate field count are within tolerance; the information per spike, the high-rate field count and the gain changes are outside.

## figs2-track-distribution, fig3a-plastic-specificity

> Rigid and plastic cells' place fields are similarly distributed along the track, both enriched near the start and goal ends.
> On the novel maze plastic cells have higher place specificity than rigid cells and fewer place fields.

In [ ]:
for DL in KS:
    use(DL); print(f"-- groups of reading {DL}")
    a, b = RIG.peak_frac.dropna(), PLA.peak_frac.dropna()
    print(f"KS p {stats.ks_2samp(a, b).pvalue:.3g}; rank-sum on distance to nearer end p "
          f"{stats.ranksums(np.minimum(a, 1 - a), np.minimum(b, 1 - b)).pvalue:.3g} (n {len(a)}, {len(b)})")
    for lab, v in [("rigid", a), ("plastic", b), ("all place cells", PCU.peak_frac.dropna())]:
        print(f"{lab}: fraction of peaks in the outer fifths {np.mean((v < 0.2) | (v > 0.8)):.3f} (uniform 0.4); median distance to nearer end {np.median(np.minimum(v, 1 - v)):.3f}")
use("D-a")

**Result: reproduced.** Neither test finds a difference between rigid and plastic peak positions, with the groups of D-a (KS p 0.484; rank-sum on the distance to the nearer end p 0.0991) or of D-b (p 0.833 and 0.264). Both groups are enriched at the ends: 0.617 of rigid and 0.708 of plastic peaks are in the outer fifths under D-a (0.596 and 0.692 under D-b), against 0.4 for a uniform spread.

In [ ]:
for DL in KS:
    use(DL); print(f"-- groups of reading {DL}")
    noPREonly = UNITS[~(UNITS.sig_PRE & ~UNITS.sig_POST)]
    for col in ["inout", "info_spike", "info_sec", "coherence", "stab_a", "stab_b", "n_fields"]:
        m1, m2, p, n1, n2 = cmp(col)
        v1, v2, pv, *_ = cmp(col, noPREonly[noPREonly.group == "rigid"], noPREonly[noPREonly.group == "plastic"])
        print(f"{col:10s} rigid median {m1:.3f}, plastic {m2:.3f}, p {p:.3g} (n {n1}, {n2}); mean rigid {RIG[col].mean():.3f} plastic {PLA[col].mean():.3f}; "
              f"without PRE-only cells p {pv:.3g}")
use("D-a")

**Result: partially reproduced.** Plastic cells are higher on all five specificity measures and have fewer fields on average under both readings of D, but not every difference reaches p < 0.05. With the D-a groups four of the six do: in/out ratio (p 0.0026), information per spike (p 0.00248), information per second (p 0.0127) and stability (reading a p 0.0125, reading b p 0.0301); coherence (p 0.209) and the number of fields (mean 1.308 against 1.475, p 0.0641) do not. With the D-b groups two do (information per second p 0.034, stability reading a p 0.0407). Without the PRE-only cells, two of six reach p < 0.05 under D-a.

## fig3b-plastic-improve and figs13-outfield-rate-falls

> Plastic but not rigid cells steadily increased their spatial specificity during maze running.
> Neither group's overall firing rate changed significantly over laps, and neither group's within-field firing rate increased significantly; plastic cells' increase in specificity came from a fall in their out-of-field firing rate, which rigid cells did not show.

In [ ]:
LAPR = {}
for DL in KS:
    use(DL); print(f"-- groups of reading {DL}")
    lrows = []
    for rw in UNITS[UNITS.group != ""].itertuples():
        r = R[rw.session]; s = S[rw.session]; dd = r["dirs"][rw.dir]
        M = g.lap_measures(s, rw.dir, rw.cell, dd["fields"][rw.cell], r["speed"], r["v1"], LAPS[rw.session], dd["occ"].shape[0])
        M = M[M[:, 0] >= 5]
        row = {"group": rw.group, "session": rw.session, "n_laps": len(M)}
        if len(M) >= 3:
            n, nin, T, Tin = M.T
            lapn = np.arange(len(M))
            with np.errstate(invalid="ignore", divide="ignore"):
                meas = {"a_pct_in": 100 * nin / n, "b_in_over_all": (nin / Tin) / (n / T), "c_in_over_out": (nin / Tin) / ((n - nin) / (T - Tin)),
                        "rate_all": n / T, "rate_in": nin / Tin, "rate_out": (n - nin) / (T - Tin)}
            for k, v in meas.items():
                ok = np.isfinite(v)
                row[k] = stats.pearsonr(lapn[ok], v[ok])[0] if ok.sum() >= 3 and np.std(v[ok]) > 0 else np.nan
        lrows.append(row)
    LAP = pd.DataFrame(lrows)
    print("units with >= 3 laps of >= 5 spikes:", LAP.groupby("group").a_pct_in.count().to_dict(), "of", LAP.groupby("group").size().to_dict())
    for k in ["a_pct_in", "b_in_over_all", "c_in_over_out", "rate_all", "rate_in", "rate_out"]:
        a, b = LAP[LAP.group == "rigid"][k].dropna(), LAP[LAP.group == "plastic"][k].dropna()
        print(f"{k:14s} rigid median r {a.median():+.3f} (signed-rank p {stats.wilcoxon(a).pvalue:.3g}), plastic {b.median():+.3f} "
              f"(p {stats.wilcoxon(b).pvalue:.3g}); plastic > rigid rank-sum p {stats.ranksums(b, a, alternative='greater').pvalue:.3g}")
    LAPR[DL] = LAP
use("D-a"); LAP = LAPR["D-a"]

**Result: not reproduced.** Under none of the three measures do plastic cells' lap correlations exceed zero (D-a: a, median r -0.028, p 0.383; b, +0.033, p 0.76; c, +0.040, p 0.789), nor exceed rigid cells' (p 0.936, 0.83, 0.831). Rigid cells' correlations are above zero for readings a (+0.092, p 0.00572) and c (+0.084, p 0.0404). With the D-b groups no plastic correlation is above zero either (p 0.483 to 0.896).

**Result: not reproduced.** With the D-a groups the out-of-field rate falls over laps significantly in rigid cells (median r -0.103, p 0.00549) and not in plastic cells (-0.048, p 0.0648), the reverse of the claim; neither group's overall rate (p 0.134 and 0.165) or within-field rate (p 0.818 and 0.184) changes significantly, as the claim says. With the D-b groups neither group's out-of-field fall is significant (rigid p 0.0782, plastic p 0.0627), and plastic cells' overall rate falls significantly (p 0.0353).

## figs16de-more-ripples-post

> Ripple events occur more often in POST than in PRE (median inter-event interval 2.47 s against 3 s), as do purely LFP-detected sharp wave-ripples (1.06 s against 1.21 s).

In [ ]:
def ieis(times, iv):
    iv = g.merge_intervals(iv)
    k = np.searchsorted(iv[:, 0], times, side="right") - 1
    ok = (k >= 0); ok[ok] = times[ok] <= iv[k[ok], 1]
    t_, k_ = times[ok], k[ok]
    o = np.argsort(t_); t_, k_ = t_[o], k_[o]
    d = np.diff(t_)[k_[1:] == k_[:-1]]
    return d[d < 60]
for Y in g.READINGS_Y:
    out = {}
    for ep in ["PRE", "POST"]:
        rip, lfa, lfb = [], [], []
        for nm, r in R.items():
            iv = g.intersect_intervals(r["off"]["S-a"][ep], r["yoke"][Y][ep])
            e = ET[(ET.session == nm) & g.reading_mask(ET, Y, "L-a", "S-a") & (ET.epoch == ep)]
            rip.append(ieis(e.mid.values, iv))
            pk, pz = r["lfp"]["peak_t"], r["lfp"]["peak_z"]
            lfa.append(ieis(pk, iv)); lfb.append(ieis(pk[pz >= 5], iv))
        out[ep] = [np.concatenate(x) for x in (rip, lfa, lfb)]
    for i, lab in enumerate(["ripple events", "LFP-SWR L-a (3 SD)", "LFP-SWR L-b (5 SD)"]):
        a, b = out["PRE"][i], out["POST"][i]
        print(f"{Y} {lab:20s} PRE median {np.median(a):.3f} s, POST {np.median(b):.3f} s, rank-sum p {stats.ranksums(a, b).pvalue:.3g} (n {len(a)}, {len(b)})")

**Result: partially reproduced, as verified; a change to not reproduced is proposed.** Intervals are shorter in POST for ripple events (2.424 against 1.944 s, p 2.37e-62) and for LFP-detected ripples under both thresholds (L-a 0.939 against 0.625 s; L-b 2.232 against 1.234 s), under every yoking. The ripple-event medians are outside tolerance (3 and 2.47 s expected), and the LFP medians match the paper's (1.21 and 1.06 s) under neither threshold. The criterion has no partial outcome: it is reproduced only when the ripple-event medians are within tolerance as well, and otherwise not reproduced. The outcome agreed in round 1 is kept and the change stays proposed: the third verification round did not agree yet, because the ripple events here are those of readings L-a and S-a only, and L-b keeps about 60% of the PRE and 72% of the POST ripple events (cell claim:figs5e-duration-unchanged), which lengthens their intervals. Whether the medians reach tolerance under L-b or S-b has not been run.

## Coactivity: figs17-coactivity-preservation, figs17d-rigid-plastic-coactivity, fig4c-*

In [ ]:
def corr3(P_):
    n = len(P_)
    r_pr = stats.pearsonr(P_.PRE, P_.RUN); r_rp = stats.pearsonr(P_.RUN, P_.POST); r_pp = stats.pearsonr(P_.PRE, P_.POST)
    return n, r_pr, r_rp, r_pp
n, r_pr, r_rp, r_pp = corr3(PAIRS)
print(f"all place-cell pairs n {n}: PRE-RUN r {r_pr[0]:.3f} (p {r_pr[1]:.3g}), RUN-POST r {r_rp[0]:.3f} (p {r_rp[1]:.3g}), PRE-POST r {r_pp[0]:.3f} (p {r_pp[1]:.3g})")
print(f"corr(RUN,POST)-corr(RUN,PRE) = {r_rp[0] - r_pr[0]:.3f}: independent Fisher z p {g.fisher_z_independent(r_rp[0], n, r_pr[0], n)[1]:.3g}; "
      f"dependent (Steiger) p {g.fisher_z_dependent(r_rp[0], r_pr[0], r_pp[0], n)[1]:.3g}")
print(f"corr(PRE,POST)-corr(RUN,POST) = {r_pp[0] - r_rp[0]:.3f}: independent p {g.fisher_z_independent(r_pp[0], n, r_rp[0], n)[1]:.3g}; "
      f"dependent p {g.fisher_z_dependent(r_pp[0], r_rp[0], r_pr[0], n)[1]:.3g}")
for nm, gx in PAIRS.groupby("session"):
    n_, a_, b_, c_ = corr3(gx)
    print(f"  {nm}: n {n_}, PRE-RUN {a_[0]:.3f}, RUN-POST {b_[0]:.3f}, PRE-POST {c_[0]:.3f}")

**Result: partially reproduced.** The order holds: PRE-RUN r 0.094 (p 6.35e-23), RUN-POST 0.267, PRE-POST 0.393, with both differences significant under both forms of the Fisher test (0.173, p 1.9e-40 independent; 0.126, p 6.95e-26). All five values are outside tolerance (0.045, 0.16, 0.34, 0.11, 0.18 expected).

In [ ]:
for DL in KS:
    print(f"-- groups of reading {DL}")
    RR = PAIRS[(PAIRS[f"g1_{DL}"] == "rigid") & (PAIRS[f"g2_{DL}"] == "rigid")]; PP = PAIRS[(PAIRS[f"g1_{DL}"] == "plastic") & (PAIRS[f"g2_{DL}"] == "plastic")]
    nr, a1, b1, c1 = corr3(RR); npl, a2, b2, c2 = corr3(PP)
    print(f"rigid pairs n {nr}: PRE-RUN {a1[0]:.3f}, RUN-POST {b1[0]:.3f}, PRE-POST {c1[0]:.3f}")
    print(f"plastic pairs n {npl}: PRE-RUN {a2[0]:.3f}, RUN-POST {b2[0]:.3f}, PRE-POST {c2[0]:.3f}")
    for lab, x1, x2 in [("PRE-RUN rigid > plastic", a1[0], a2[0]), ("PRE-POST rigid > plastic", c1[0], c2[0]), ("RUN-POST plastic > rigid", b2[0], b1[0])]:
        nn = (nr, npl) if "rigid >" in lab else (npl, nr)
        z, p = g.fisher_z_independent(x1, nn[0], x2, nn[1])
        print(f"{lab}: z {z:.2f}, p {p:.3g}")

**Result: not reproduced.** With the D-a groups rigid pairs preserve PRE coactivity into POST more than plastic pairs (0.440 against 0.203, p 7.44e-06), and plastic pairs preserve RUN coactivity into POST more (0.324 against 0.208, p 0.0362), but the rigid PRE-RUN preservation (0.118 against 0.002) is not significantly larger (p 0.051), and the criterion asks for all three differences. With the D-b groups only the PRE-POST difference is significant (p 0.00645; PRE-RUN p 0.189, RUN-POST p 0.0553). The plastic PRE-RUN value is within its tolerance; the others are outside or at the edge.

In [ ]:
def pcorr(x, y, z):
    rxy, rxz, ryz = np.corrcoef(x, y)[0, 1], np.corrcoef(x, z)[0, 1], np.corrcoef(y, z)[0, 1]
    return (rxy - rxz * ryz) / np.sqrt((1 - rxz ** 2) * (1 - ryz ** 2))
def windows(rate_col, ranking):
    cr = CELL[[rate_col]].copy()
    pcs = {(nm, u) for nm, (p_, _) in VEC.items() for u in p_}
    cr = cr[[k in pcs for k in cr.index]]
    if ranking == "a":
        cr["pct"] = cr.groupby(level=0)[rate_col].rank(pct=True)
    else:
        cr["pct"] = cr[rate_col].rank(pct=True)
    pct = cr.pct.to_dict()
    P_ = PAIRS.copy()
    P_["p1"] = [pct[(a, b)] for a, b in zip(P_.session, P_.c1)]
    P_["p2"] = [pct[(a, b)] for a, b in zip(P_.session, P_.c2)]
    rows = []
    for w in range(0, 81):
        lo_, hi_ = w / 100, (w + 20) / 100
        m = (P_.p1 > lo_) & (P_.p1 <= hi_) & (P_.p2 > lo_) & (P_.p2 <= hi_)
        x = P_[m]
        if ranking == "a":
            vals = [(gx.PRE.mean(), gx.POST.mean(), gx.dco.mean(), pcorr(gx.RUN, gx.POST, gx.PRE) if len(gx) > 3 else np.nan)
                    for _, gx in x.groupby("session")]
            vals = np.nanmean(np.array(vals, float), 0) if vals else [np.nan] * 4
        else:
            vals = (x.PRE.mean(), x.POST.mean(), x.dco.mean(), pcorr(x.RUN, x.POST, x.PRE) if len(x) > 3 else np.nan)
        rows.append({"w": w, "n": len(x), "PRE": vals[0], "POST": vals[1], "dco": vals[2], "pcorr": vals[3], "pairs": x})
    return pd.DataFrame(rows)
# session firing rate under the three readings of fig4a-rigid-faster (verifier round 1); sess_c is the whole recording
rows = []
for rate_for_windows in ["sess_a", "sess_b", "sess_c"]:
    for ranking in ["a", "b"]:
        W = windows(rate_for_windows, ranking)
        rho1 = stats.spearmanr(W.w, W.PRE); rho2 = stats.spearmanr(W.w, W.dco); rho3 = stats.spearmanr(W.w, W.pcorr, nan_policy="omit")
        fast, slow = W.iloc[-1].pairs, W.iloc[0].pairs
        rows.append({"rate": rate_for_windows, "ranking": ranking, "PRE_rho": rho1[0], "PRE_p": rho1[1], "fast_change": fast.dco.mean(),
                     "fast_p": stats.wilcoxon(fast.dco).pvalue, "slow_change": slow.dco.mean(), "change_rho": rho2[0], "change_p": rho2[1],
                     "pcorr_rho": rho3[0], "pcorr_p": rho3[1], "pcorr_slowest": W.pcorr.iloc[0], "pcorr_fastest": W.pcorr.iloc[-1],
                     "pcorr_argmax": int(W.pcorr.idxmax()), "n_pairs": f"{W.n.min()}-{W.n.max()}"})
F4c = pd.DataFrame(rows)
print(F4c.to_string(float_format=lambda v: f"{v:.3g}"))

# Figure S16C: low-rate half against high-rate half (cells of rate-split-medians' split)
lowc = {(rw.session, rw.cell) for rw in PCU[PCU.rgroup == "low"].itertuples()}
highc = {(rw.session, rw.cell) for rw in PCU[PCU.rgroup == "high"].itertuples()}
lowc, highc = lowc - highc, highc - lowc
LL = PAIRS[[((a, b) in lowc) and ((a, c) in lowc) for a, b, c in zip(PAIRS.session, PAIRS.c1, PAIRS.c2)]]
HH = PAIRS[[((a, b) in highc) and ((a, c) in highc) for a, b, c in zip(PAIRS.session, PAIRS.c1, PAIRS.c2)]]
print(f"low-rate pairs n {len(LL)} change {LL.dco.mean():+.4f}; high-rate pairs n {len(HH)} change {HH.dco.mean():+.4f}; "
      f"low > high rank-sum p {stats.ranksums(LL.dco, HH.dco, alternative='greater').pvalue:.3g}; high-rate change signed-rank p {stats.wilcoxon(HH.dco).pvalue:.3g}")

**Result: reproduced, resting on the whole-recording rate (sess_c).** With the whole-recording rate, the "overall session (SESS) firing rates" of the Figure 4B legend, PRE coactivity rises with the window's rate rank under both rankings (Spearman rho 0.867 and 0.808, p below 1e-19), and the POST - PRE change in the fastest window is not significant (ranking a: -0.0010, p 0.572; ranking b: +0.0035, p 0.172). With the offline rates of fig4a's readings a and b the PRE rise holds (rho 0.797 to 0.850), and the fastest window's change is significant once, under reading b with pooled ranks (p 0.0303).

**Result: not reproduced.** Under every session-rate reading the low-rate half's change in coactivity is not larger than the high-rate half's (0.0059 against 0.0053, p 0.205; the halves are those of rate-split-medians). The change falls with rate rank under both rankings with rate reading a (p 0.0442 and 5.15e-05), only under pooled ranks with reading b (p 0.658 and 2.08e-06), and only under ranks within session with the whole-recording rate (p 0.02 and 0.126).

**Result: not reproduced, resting on the whole-recording rate (sess_c).** With the whole-recording rate the partial correlation falls with rate rank under pooled ranks (rho -0.756, p 3.25e-16, peak at window 13) but not under ranks within session (rho -0.055, p 0.624; peak at window 58), and the criterion asks for both. With the offline PRE and POST rate (reading b) it falls under both rankings (rho -0.499 and -0.839) and peaks in slow windows (13 and 15); with reading a it falls under both (rho -0.401 and -0.849) but peaks at window 62 under ranks within session.

## figs18-temporal-bias

> Pair-wise temporal biases are not preserved from PRE to RUN (r = -0.009) but are preserved from RUN to POST (r = 0.117) and PRE to POST (r = 0.201); the PRE to POST preservation is carried by rigid cell pairs (r = 0.29) and the RUN to POST preservation by plastic cell pairs (r = 0.26).

In [ ]:
brow = []
for nm, (pcs, ivs) in VEC.items():
    s = S[nm]
    for i in range(len(pcs)):
        for j in range(i + 1, len(pcs)):
            if pcs[i] // 100 == pcs[j] // 100:
                continue
            row = {"session": nm, "c1": pcs[i], "c2": pcs[j]}
            for ep in ["PRE", "RUN", "POST"]:
                row[ep], row["n_" + ep] = g.ccg_bias(s["spikes"][pcs[i]], s["spikes"][pcs[j]], ivs[ep])
            brow.append(row)
BIAS = pd.DataFrame(brow)
BIAS["g1"] = [cgroup.get((a, b), "") for a, b in zip(BIAS.session, BIAS.c1)]
BIAS["g2"] = [cgroup.get((a, b), "") for a, b in zip(BIAS.session, BIAS.c2)]
def bias_r(df, e1, e2, rng, n=1000):
    m = (df["n_" + e1] >= 40) & (df["n_" + e2] >= 40)
    x, y = df[m][e1].values, df[m][e2].values
    sg = rng.choice([-1, 1], (n, len(x)))
    X, Yv = sg * x, sg * y
    X = X - X.mean(1, keepdims=True); Yv = Yv - Yv.mean(1, keepdims=True)
    rs = (X * Yv).sum(1) / np.sqrt((X ** 2).sum(1) * (Yv ** 2).sum(1))
    tt = rs * np.sqrt((len(x) - 2) / (1 - rs ** 2))
    ps = 2 * stats.t.sf(np.abs(tt), len(x) - 2)
    return np.median(rs), np.median(ps), len(x)
rows = []
for k in [0, 1000, 2000, 3000, 4000]:
    rng = np.random.default_rng(seed("figs18-temporal-bias") + k)
    row = {"seed": f"+{k}"}
    for lab, df in [("all", BIAS), ("rigid", BIAS[(BIAS.g1 == "rigid") & (BIAS.g2 == "rigid")]), ("plastic", BIAS[(BIAS.g1 == "plastic") & (BIAS.g2 == "plastic")])]:
        for e1, e2 in [("PRE", "RUN"), ("RUN", "POST"), ("PRE", "POST")]:
            r_, p_, n_ = bias_r(df, e1, e2, rng, 1000 if lab == "all" else 1000)
            row[f"{lab} {e1}-{e2} r"] = r_; row[f"{lab} {e1}-{e2} p"] = p_; row[f"{lab} {e1}-{e2} n"] = n_
    rows.append(row)
    if k == 0:
        print(pd.Series(row).to_string())
FB = pd.DataFrame(rows)
print(FB[[c for c in FB.columns if c.endswith(" r")]].round(3).to_string())
# groups of reading D-b, claim seed
BIAS["g1b"] = [CG["D-b"].get((a, b), "") for a, b in zip(BIAS.session, BIAS.c1)]
BIAS["g2b"] = [CG["D-b"].get((a, b), "") for a, b in zip(BIAS.session, BIAS.c2)]
rng = np.random.default_rng(seed("figs18-temporal-bias"))
for lab, df in [("rigid", BIAS[(BIAS.g1b == "rigid") & (BIAS.g2b == "rigid")]), ("plastic", BIAS[(BIAS.g1b == "plastic") & (BIAS.g2b == "plastic")])]:
    for e1, e2 in [("PRE", "RUN"), ("RUN", "POST"), ("PRE", "POST")]:
        r_, p_, n_ = bias_r(df, e1, e2, rng)
        print(f"D-b {lab} {e1}-{e2}: r {r_:.3f}, p {p_:.3g}, n {n_}")
print(f"{time.time() - T0:.0f} s")

**Result: not reproduced.** All pairs: PRE-RUN r 0.007 (p 0.50), RUN-POST 0.119 and PRE-POST 0.221 (p ~ 0), as the claim says. With the D-a groups rigid PRE-POST 0.300 and plastic RUN-POST 0.193 are significant and plastic PRE-POST (0.071, p 0.111) is not, but rigid RUN-POST (0.131, p 0.00127) is significant where the claim has it not, so the significance pattern of the criterion fails. With the D-b groups rigid RUN-POST is significant too (0.127, p 0.00626). Four of the five values are within tolerance; plastic RUN-POST (0.193 against 0.26) is not. The five seeds give the same r to three decimals.

## fig4d-slow-sequences and fig4e-gain-sequences

Replay rerun with templates of only the low-rate or only the high-rate place cells (split of rate-split-medians), and of only the low-gain or high-gain cells (within-ripple gain, PRE and POST together, split the same way).

In [ ]:
# fig4d-slow-sequences and fig4e-gain-sequences: templates of only the low or high half; the halves do not
# depend on Y or D, so each template is scored once on all rows and every reading is a subset.
rng_rr = np.random.default_rng(seed("fig4d-slow-sequences"))
rng_b = np.random.default_rng(seed("fig4d-slow-sequences", 1))
PCU["gain_pp"] = PCU.sessgain_b
PCU["ggroup"] = split(PCU, "gain_pp", SPLIT[0])
rows = []
for lab, colg in [("rate", "rgroup"), ("gain", "ggroup")]:
    for half in ["low", "high"]:
        keepset = {(rw.session, rw.dir, rw.cell) for rw in PCU[PCU[colg] == half].itertuples()}


        out = []
        for (nm, d), p in PREP.items():
            pcs = p["cells"]; kidx = [i for i, u in enumerate(pcs) if (nm, d, u) in keepset]
            ev, rz, _, _, _ = g.rescore_batch(p, kidx, rng_rr, N_RERUN)
            out.append(pd.DataFrame({"session": nm, "dir": d, "row": ev, "epoch": p["epoch"][ev], "rz": rz,
                                     **{Y: p[Y][ev] for Y in g.READINGS_Y}, "D-b": p["D-b"][ev]}))
        dfa = pd.concat(out, ignore_index=True)
        for Y in g.READINGS_Y:
            for D in g.READINGS_D:
                x = dfa[dfa[Y] & (dfa["D-b"] if D == "D-b" else True)]
                a, b = x[x.epoch == "PRE"].rz.values, x[x.epoch == "POST"].rz.values
                ci = g.boot_diff_ci(b, a, rng_b)
                rows.append({"split": lab, "half": half, "Y": Y, "D": D, "POST-PRE": b.mean() - a.mean(), "ci_lo": ci[0], "ci_hi": ci[1],
                             "ranksum_p_POST>PRE": stats.ranksums(b, a, alternative="greater").pvalue, "n_PRE": len(a), "n_POST": len(b)})
F4 = pd.DataFrame(rows)
print(F4.to_string(float_format=lambda v: f"{v:.3g}"))

print(f"{time.time() - T0:.0f} s, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

**Result: reproduced.** With only low-rate cells POST - PRE rZ is above zero under all six readings of Y and D (main reading 0.046, interval 0.016 to 0.077, rank-sum p 0.0066 to 0.0068; up to 0.076 to 0.077 under Y-c with D-b), and with only high-rate cells it is not under any (main reading -0.00306, interval -0.0262 to 0.019, p 0.638; smallest p-value 0.144). Both test readings agree. The replay reruns give values that differ in the third significant digit between executions of the same notebook, so the values that do are given here to two significant digits.

**Result: not reproduced.** With only high-gain cells POST - PRE rZ is above zero under all six readings (main reading 0.062 to 0.063, interval 0.037 to 0.085, p 1.4e-06), but the low-gain cells also increase, less, under five of them at rank-sum p < 0.05 (main reading 0.032, interval 0.0061 to 0.057, p 0.016); under Y-a with D-b the low-gain increase is not significant (p 0.163). The criterion has no partial outcome of its own ("As fig4d-slow-sequences" names the method, and the criterion then gives its own rule): it is reproduced only when the low-gain group's POST - PRE rZ is not above zero, and here it is under five of six readings, so the result is not reproduced. The third verification round agreed with this outcome. The replay reruns give values that differ in the third significant digit between executions of the same notebook, so the values that do are given here to two significant digits.

## figs11-cluster-quality-similar

> Rigid and plastic cells have similar waveforms and similar cluster quality and stability across PRE, MAZE and POST, so clustering errors do not explain their differences.

The PCA features (`.fet`) and cluster ids (`.clu`) are in the per-session tarballs of hc-11 (0.8 to 2.3 GB each), streamed here. Unit ids are shank x 100 + cluster id; the check is that each cluster's row count equals the unit's spike count and the `.res` times equal its spike times. Quality and stability as in the helper (`session_cluster_quality`): the cluster centre and covariance are those of all of the unit's spikes, and stability is the mean of the two medians of each condition's spikes' distance to the other condition's centre (assumptions). Waveforms (the `_spk` tarballs, 4.8 to 14.7 GB each) are not read.

In [ ]:
# Spike features and cluster ids are streamed from the per-session tarballs of hc-11 (<session>.tar.gz), four
# sessions at a time. Units: every place cell. Conditions (Figure S11 legend): PRE ripple events and POST ripple
# events of the main reading (Y-a, L-a, S-a), and MAZE running at >= 5 cm/s.
import re
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp
jobs = []
for nm, r in R.items():
    s = S[nm]
    e = ET[(ET.session == nm) & g.reading_mask(ET, "Y-a", "L-a", "S-a")]
    civ = {"PRE": e[e.epoch == "PRE"][["t0", "t1"]].values, "MAZE": VEC[nm][1]["RUN"], "POST": e[e.epoch == "POST"][["t0", "t1"]].values}
    units = sorted({u for dd in r["dirs"].values() for u in dd["place_cells"]})
    jobs.append((nm, {u: s["spikes"][u] for u in units}, units, civ))
t1 = time.time()
with ProcessPoolExecutor(4, mp_context=mp.get_context("spawn")) as ex:
    res = list(ex.map(g._cq_worker, jobs))
print(f"cluster quality: 8 tarballs streamed in {time.time() - t1:.0f} s")
for (nm, *_), (df, ck) in zip(jobs, res):
    mem = ck["members"]
    kinds = {k: sum(1 for b, _ in mem if re.search(rf"\.{k}\.\d+$", b)) for k in ["clu", "fet", "res"]}
    print(f"{nm}: {len(mem)} members ({kinds['clu']} .clu, {kinds['fet']} .fet, {kinds['res']} .res, .fet {sum(sz for b, sz in mem if '.fet.' in b) / 1e9:.2f} GB); "
          f"units whose cluster row count differs from the spike count {ck['count_mismatch']}; .fet and .clu rows differ on {ck['fet_rows_mismatch']}; "
          f"max |.res time - spike time| {ck['res_maxerr']:.2g} s; PCA columns {sorted(df.n_pca.dropna().unique().astype(int).tolist())}")
print("first members of the first tarball:", res[0][1]["members"][:4])
CQ = pd.concat([x[0] for x in res], ignore_index=True)
print(CQ.drop(columns=["cell"]).describe().T[["count", "mean", "50%", "min", "max"]].round(3).to_string())
rows = []
for DL, K_ in KS.items():
    U = K_[K_.group != ""].merge(CQ, on=["session", "cell"], how="left")
    for col in ["q_PRE", "q_MAZE", "q_POST", "stab_PRE_MAZE", "stab_MAZE_POST", "stab_PRE_POST"]:
        a, b = U[U.group == "rigid"][col].dropna(), U[U.group == "plastic"][col].dropna()
        rows.append({"D": DL, "measure": col, "rigid_median": a.median(), "plastic_median": b.median(), "difference": b.median() - a.median(),
                     "ranksum_p": stats.ranksums(a, b).pvalue, "n_rigid": len(a), "n_plastic": len(b)})
F11 = pd.DataFrame(rows)
print(F11.to_string(float_format=lambda v: f"{v:.3g}"))
print("comparisons with p < 0.05:", F11.groupby("D").ranksum_p.apply(lambda p: int((p < 0.05).sum())).to_dict())
print(f"{time.time() - T0:.0f} s, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

In [ ]:
# Room for a difference (verifier round 2). With each unit's own centre and covariance, the distance of a
# Gaussian cluster's spikes follows a chi distribution with as many degrees of freedom as PCA columns, so the
# median quality is near its median (5.42 for 30 columns, 4.83 for 24) by construction. Shown: the median of
# each measure over that reference, its spread across units, and a bootstrap 95% interval of the plastic minus
# rigid difference of medians, raw and divided by the reference.
from scipy.stats import chi
rng_q = np.random.default_rng(seed("figs11-cluster-quality-similar"))
CQ["ref"] = chi.median(CQ.n_pca)
print("chi median for 24 and 30 columns:", round(chi.median(24), 3), round(chi.median(30), 3))
rows = []
for DL, K_ in KS.items():
    U = K_[K_.group != ""].merge(CQ, on=["session", "cell"], how="left")
    for col in ["q_PRE", "q_MAZE", "q_POST", "stab_PRE_MAZE", "stab_MAZE_POST", "stab_PRE_POST"]:
        x = U[["group", col, "ref"]].dropna()
        x = x.assign(rel=x[col] / x.ref)
        a, b = x[x.group == "rigid"], x[x.group == "plastic"]
        bd = [np.median(rng_q.choice(b[col].values, len(b))) - np.median(rng_q.choice(a[col].values, len(a))) for _ in range(2000)]
        br = [np.median(rng_q.choice(b.rel.values, len(b))) - np.median(rng_q.choice(a.rel.values, len(a))) for _ in range(2000)]
        q1, q3 = np.percentile(x[col], [25, 75])
        rows.append({"D": DL, "measure": col, "median_over_ref": x.rel.median(), "IQR_units": q3 - q1,
                     "diff": b[col].median() - a[col].median(), "diff_lo": np.percentile(bd, 2.5), "diff_hi": np.percentile(bd, 97.5),
                     "rel_diff": b.rel.median() - a.rel.median(), "rel_lo": np.percentile(br, 2.5), "rel_hi": np.percentile(br, 97.5),
                     "rel_ranksum_p": stats.ranksums(a.rel, b.rel).pvalue})
print(pd.DataFrame(rows).to_string(float_format=lambda v: f"{v:.3g}"))


**Result: reproduced, resting on the definitions of the cluster covariance and of stability.** The tarballs stream with every unit's row count equal to its spike count and the `.res` times equal to the spike times. With the groups of D-a (and of D-b) none of the six rank-sum comparisons of rigid against plastic cells is significant: cluster quality in PRE ripple events, MAZE running and POST ripple events, and stability between each pair of conditions (D-a medians 5.43 to 5.72 in both groups, differences under 0.04, p 0.205 to 0.914; D-b p 0.124 to 0.926). The waveforms of Figure S11A were not tested. The measure has little room to differ. With each unit's own centre and covariance, the median distance of a Gaussian cluster's spikes is the median of a chi distribution with as many degrees of freedom as PCA columns (5.42 for 30, 4.83 for 24), and the medians here are 1.02 to 1.06 times it (cell `figs11-effect-size`). The bootstrap 95% intervals of the plastic minus rigid difference of medians lie within -0.135 to 0.073, and, divided by the reference, within -0.020 to 0.013: the tests exclude differences of more than about 2.5% of the median, where the measure exceeds its reference by 2 to 6%. The outcome follows the criterion, but a claim of absence on this measure says little.

## Summary

| Claim | Outcome |
|---|---|
| sessions-and-tracks | reproduced |
| place-cell-count | not reproduced (562 pyramidal cells; 573 place cells against 491 with the kernel read as its SD; 507 with it read as its full width) |
| fig1c-above-chance | reproduced |
| fig1c-post-gt-pre | reproduced |
| fig1c-maze-gt-pre | reproduced |
| figs5b-abs-r-above-null | reproduced |
| figs5e-duration-unchanged | not reproduced (rests on yoking Y-a; 180 against 176 ms, p 0.000531) |
| figs5f-more-participation-post | reproduced |
| figs9b-pcc-post-higher | partially reproduced (tests hold, medians outside tolerance) |
| fig2b-strong-contributors | not reproduced (rests on D-a and on the kernel as its SD: 246 cells; D-b gives 217; the kernel as its width gives 232, 102, 200 and 70, all within tolerance) |
| fig2-pre-post-overlap | reproduced (rests on D-a: 71.7%; D-b gives 54.1%; the kernel as its width 68.6%) |
| rigid-plastic-counts | not reproduced (rests on D-a and on the kernel as its SD: 120 and 120; D-b gives 104 and 104; the kernel as its width 112 and 112) |
| figs19c-pcc-change-per-cell | not reproduced |
| fig2d-shuffle-high-dpcc | reproduced (rests on Y-a and on reading (a) of eliminated: at 10% under five of six readings, at 5% under Y-c with D-b; at 5% under all six with reading (b)) |
| figs12-high-dpcc-exclusion | reproduced (all six readings) |
| fig2e-remove-plastic | reproduced (all six readings) |
| figs2-track-distribution | reproduced |
| fig3a-plastic-specificity | partially reproduced |
| fig3b-plastic-improve | not reproduced |
| figs13-outfield-rate-falls | not reproduced |
| fig4a-rigid-faster | not reproduced (PRE not significant) |
| figs14-plastic-ripple-gain | not reproduced (PRE not significant) |
| figs14-plastic-ripple-bursting | not reproduced |
| figs16ab-plastic-synchrony-increase | not reproduced (rests on D-a; holds with the D-b groups) |
| figs11-cluster-quality-similar | reproduced (no difference in cluster quality or stability, on a measure near its chi reference by construction; waveforms not tested) |
| rate-split-medians | reproduced |
| figs16c-low-rate-coding | partially reproduced |
| fig4c-pre-fast-coactivation | reproduced (rests on the whole-recording rate) |
| fig4c-slow-coactivation-increase | not reproduced |
| fig4c-slow-replay | not reproduced (rests on the whole-recording rate; holds with the offline PRE and POST rate) |
| fig4d-slow-sequences | reproduced (all six readings) |
| fig4e-gain-sequences | not reproduced (the low-gain group also rises, under five of six readings) |
| figs16de-more-ripples-post | partially reproduced (a change to not reproduced stays proposed: the ripple-event medians are outside tolerance under L-a and S-a, and L-b and S-b were not run for them) |
| figs17-coactivity-preservation | partially reproduced |
| figs17d-rigid-plastic-coactivity | not reproduced (PRE-RUN difference p 0.051) |
| figs18-temporal-bias | not reproduced (rigid RUN-POST significant) |

Outcomes that turn on one assumption, each flipping under the other reading: figs5e-duration-unchanged and fig2d-shuffle-high-dpcc on the yoking Y-a; fig4c-pre-fast-coactivation and fig4c-slow-replay on the whole-recording rate; fig2b-strong-contributors, fig2-pre-post-overlap, rigid-plastic-counts and figs16ab-plastic-synchrony-increase on the direction reading D-a; fig2b-strong-contributors and rigid-plastic-counts also on the kernel read as its SD.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- 200 null templates per event and 100 cell-specific shuffles for the PCC, where the paper uses 1,000 of each. On Buddy_06272013 rescored with 1,000 of each, rZ correlates at 0.996 and PCC at 0.995 with the reduced counts, and 40 of 41 units keep their strongly contributing label (39 their rigid or plastic label) under D-a. The reruns use 200 null templates (Figures 2D, 4D and 4E), 100 (Figure 2E) or 50 (Figure S12). The replay pipeline is run under the claims' seeds only; the five-seed check covers the bootstraps and the temporal-bias sign flips.
- Figure 2D: each percentage is tested with 100 shuffled templates, each event's rZ averaged over them, against the null of the intact template; the paper's "templates (n = 1,000 each)" does not say whether 1,000 shuffled templates or 1,000 null templates are meant.
- Open choices taken as assumptions (helper docstrings): offline states from the lab's NREM, Intermediate and Drowsy epochs (S-a), the buzcode NREM epochs as reading S-b; on the maze, immobility below 5 cm/s; the LFP channel; the 1D position as the track outside the reward areas; laps from the stretches of defined 1D position; the 5 cm kernel as its SD (the claims file's reading); the main reading Y-a, L-a, S-a, D-a and P-a for the claims that depend on the rigid and plastic split, with D-b shown for each; the whole-recording rate as the session firing rate of Figure 4B and 4C; for Figure S11 the cluster covariance of all spikes and stability as the mean of the two cross-condition medians.
- The pooled null of Figure S5B is 20 null |r| values kept per event.
- The values of the replay reruns (Figures 2D, 2E, 4D, 4E and S12) were seen to differ in the third significant digit between two executions of the same notebook (fig4d main reading 0.0458 and 0.046), which the seeded draws do not explain; the outcomes are the same in both.
- Figure S11: cluster quality and stability are tested; the waveforms (Figure S11A) are not.

What was found about the datasets:

- DANDI:000044 and CRCNS hc-11 hold the same eight sessions: unit ids, cell types, every spike time and the lab's state epochs are identical. They are counted once.
- Gatsby_08022013: the sessInfo `sessDuration` and the POST epoch end at 30413.6 s, but spikes (263,976 of them) and the LFP continue to 32002.4 s (cell `finding-spikes-after-session-end`).
- Position is sampled at 39.06 Hz, where the Methods state 39.69 Hz (cell `finding-position-rate`).
- The counts of pyramidal cells (562) and interneurons (128), the four rats, eight sessions and the mazes agree with the paper. The LFP and the spikes share a clock (events hold LFP ripple peaks; shifted peaks fall in 8 to 14% of events), and so do position and spikes (decoding error 0.04 to 0.22 m, against 0.44 to 0.92 m with the position shifted 30 s). Position is tracked only within the maze epoch.
- The hc-11 feature tarballs agree with the sessInfo spikes: unit ids are shank x 100 + cluster id, every unit's cluster row count equals its spike count, and the `.res` times equal its spike times.